In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import math
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency

from docx import Document
from docx.shared import Inches

import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

<h1>Preliminary Phase (100)</h1>
<p>Have data frrom all sites...</p>

In [2]:
# Preliminary phase data
df_gh_prelim = pd.read_csv('../data/raw/ghana/raw/prelims.csv')
df_ml_prelim = pd.read_csv('../data/raw/mali/raw/prelims.csv', sep=',')
df_tn_prelim = pd.read_csv('../data/raw/tanzania/raw/tanzania_prelim.csv')
df_ug_prelim = pd.read_csv('../data/raw/uganda/raw/prelims.csv')
df_zim_prelim = pd.read_csv('../data/raw/zim_zam/raw/zim.csv')
df_zam_prelim = pd.read_csv('../data/raw/zim_zam/raw/zam.csv')
df_ng_prelim = pd.read_csv('../data/raw/nigeria/raw/prelims.csv')

print('df_gh_prelim:', df_gh_prelim.shape)
print('df_ml_prelim:', df_ml_prelim.shape)
print('df_tn_prelim:', df_tn_prelim.shape)
print('df_ug_prelim:', df_ug_prelim.shape)
print('df_zim_prelim:', df_zim_prelim.shape)
print('df_zam_prelim:', df_zam_prelim.shape)
print('df_ng_prelim:', df_ng_prelim.shape)
print(' ')
print('preliminary total:', df_gh_prelim.shape[0]+df_ml_prelim.shape[0]+df_tn_prelim.shape[0]+df_ug_prelim.shape[0]+df_zim_prelim.shape[0]+df_zam_prelim.shape[0]+df_ng_prelim.shape[0])

df_gh_prelim: (101, 82)
df_ml_prelim: (104, 219)
df_tn_prelim: (100, 52)
df_ug_prelim: (100, 94)
df_zim_prelim: (100, 14)
df_zam_prelim: (100, 9)
df_ng_prelim: (100, 202)
 
preliminary total: 705


In [3]:
df_gh_prelim['country'] = 'ghana'
df_ml_prelim['country'] = 'mali'
df_tn_prelim['country'] = 'tanzania'
df_ug_prelim['country'] = 'uganda'
df_zim_prelim['country'] = 'zimbabwe'
df_zam_prelim['country'] = 'zambia'
df_ng_prelim['country'] = 'nigeria'

In [4]:
def clean_gh_prelim(df):
    df[['dob_newborn','date_of_visit']] = df[['dob_newborn','date_of_visit']].apply(
        pd.to_datetime, format='%d %m %Y', errors='coerce'
    )
    df['Facility ID'] = 'Kumasi'
    df['confirmatory_test'] = 'ief'
    df['Study participant ID'] = 'GH' + df['record_id'].astype(str)
    df['Age_days'] = (df['date_of_visit'] - df['dob_newborn']).dt.days
    return df

df_gh_prelim = clean_gh_prelim(df_gh_prelim)

In [5]:
def clean_ml_prelim(df):
    df[['dob_newborn','date_of_visit']] = df[['dob_newborn','date_of_visit']].apply(
        lambda s: pd.to_datetime(s.astype(str).str.strip(), dayfirst=True, errors='coerce')
    )
    df['confirmatory_test'] = 'hplc'
    df = df.drop(columns=['participant_id'])
    df['Study participant ID'] = 'ML' + df['record_id'].astype(str)
    # df['Age_days'] = (df['date_of_visit'] - df['dob_newborn']).dt.days
    df['Age_days'] = 1
    return df

df_ml_prelim = clean_ml_prelim(df_ml_prelim)

In [6]:
def clean_tn_prelim(df):
    df[['dob_newborn','date_of_visit']] = df[['dob_newborn','date_of_visit']].apply(
        lambda s: pd.to_datetime(s.astype(str).str.strip(), dayfirst=True, errors='coerce')
    )
    df['confirmatory_test'] = 'ief'
    df = df.drop(columns=['participant_id'])
    df['Study participant ID'] = 'TN' + df['record_id'].astype(str)
    df['Age_days'] = (df['date_of_visit'] - df['dob_newborn']).dt.days
    return df

df_tn_prelim = clean_tn_prelim(df_tn_prelim)

In [7]:
def clean_ug_prelim(df):
    df[['dob_newborn','date_of_visit']] = df[['Date of birth (D-M-Y).1','Date of visit']].apply(
        lambda s: pd.to_datetime(s.astype(str).str.strip(), dayfirst=True, errors='coerce')
    )
    df['confirmatory_test'] = 'ief'
    df['Study participant ID'] = 'UG' + df['StudyID'].astype(str)
    df['Age_days'] = (df['date_of_visit'] - df['dob_newborn']).dt.days
    df.drop('Sex', axis=1, inplace=True)
    return df

df_ug_prelim = clean_ug_prelim(df_ug_prelim)

In [8]:
df_zim_prelim.head()

,StudyID,Facility ID,Study participant ID,Consent date,Date of birth,Age,Sex,When was the newborn screened?,Screening date (D-M-Y),Kind of screening test done (Please check all that apply),Standard POCT done?,Standard POCT Results,DBS POCT Results,IEF Results,country
0,1,PTH,NBSZIM001,1/22/2024,1/21/2024,NaN,Male,At Birthing Facility,1/22/2024,Standard PoCT,Yes,AA,AA,FA,zimbabwe
1,2,PTH,NBSZIM003,1/22/2024,1/21/2024,NaN,Female,At Birthing Facility,1/22/2024,Standard PoCT,Yes,AA,AA,FA,zimbabwe
2,3,PTH,NBSZIM002,1/22/2024,1/21/2024,NaN,Female,At Birthing Facility,1/22/2024,Standard PoCT,Yes,AA,AA,FA,zimbabwe
3,4,PTH,NBSZIM004,1/23/2024,1/22/2024,NaN,Male,At Birthing Facility,1/23/2024,Standard PoCT,Yes,AA,AA,FA,zimbabwe
4,5,PTH,NBSZIM005,1/23/2024,1/23/2024,NaN,Female,At Birthing Facility,1/23/2024,Standard PoCT,Yes,AA,AA,FA,zimbabwe


In [9]:
df_zim_prelim.columns

Index(['StudyID', 'Facility ID', 'Study participant ID', 'Consent date',
       'Date of birth', 'Age', 'Sex', 'When was the newborn screened?',
       'Screening date (D-M-Y)',
       'Kind of screening test done (Please check all that apply)',
       'Standard POCT done?', 'Standard POCT Results', ' DBS POCT Results',
       'IEF Results ', 'country'],
      dtype='object')

In [10]:
def clean_zim_prelim(df):
    df[['Date of birth','Consent date']] = df[['Date of birth','Consent date']].apply(
        lambda s: pd.to_datetime(s.astype(str).str.strip(), dayfirst=True, errors='coerce')
    )
    df['confirmatory_test'] = 'ief'
    df['Study participant ID'] = 'ZM' + df['StudyID'].astype(str)
    df['Age'] = (df['Consent date'] - df['Date of birth']).dt.days
#     df = df.drop(df.columns[-3], axis=1)
    return df

df_zim_prelim = clean_zim_prelim(df_zim_prelim)

In [11]:
df_zim_prelim.columns

Index(['StudyID', 'Facility ID', 'Study participant ID', 'Consent date',
       'Date of birth', 'Age', 'Sex', 'When was the newborn screened?',
       'Screening date (D-M-Y)',
       'Kind of screening test done (Please check all that apply)',
       'Standard POCT done?', 'Standard POCT Results', ' DBS POCT Results',
       'IEF Results ', 'country', 'confirmatory_test'],
      dtype='object')

In [12]:
df_zim_prelim.head()

,StudyID,Facility ID,Study participant ID,Consent date,Date of birth,Age,Sex,When was the newborn screened?,Screening date (D-M-Y),Kind of screening test done (Please check all that apply),Standard POCT done?,Standard POCT Results,DBS POCT Results,IEF Results,country,confirmatory_test
0,1,PTH,ZM1,2024-01-22,2024-01-21,1,Male,At Birthing Facility,1/22/2024,Standard PoCT,Yes,AA,AA,FA,zimbabwe,ief
1,2,PTH,ZM2,2024-01-22,2024-01-21,1,Female,At Birthing Facility,1/22/2024,Standard PoCT,Yes,AA,AA,FA,zimbabwe,ief
2,3,PTH,ZM3,2024-01-22,2024-01-21,1,Female,At Birthing Facility,1/22/2024,Standard PoCT,Yes,AA,AA,FA,zimbabwe,ief
3,4,PTH,ZM4,2024-01-23,2024-01-22,1,Male,At Birthing Facility,1/23/2024,Standard PoCT,Yes,AA,AA,FA,zimbabwe,ief
4,5,PTH,ZM5,2024-01-23,2024-01-23,0,Female,At Birthing Facility,1/23/2024,Standard PoCT,Yes,AA,AA,FA,zimbabwe,ief


In [13]:
df_zam_prelim.head()

,StudyID,Date standard POCT test done (D-M-Y),Date of Standard POCT results (D-M-Y),Sex,When was the newborn screened?,Standard POCT done?,Standard POCT Results,DBS POCT Results,IEF Results,country
0,106-283,10/4/2024,10/4/2024,Female,At Immunisation Facility,Yes,AS,AS,FAS - Sickle cell trait (AS),zambia
1,106-284,10/4/2024,10/4/2024,Female,At Immunisation Facility,Yes,AA,AA,FA - NO abnormal hemoglobin,zambia
2,106-285,10/2/2024,10/2/2024,Male,At Birthing Facility,Yes,AA,AA,FA - NO abnormal hemoglobin,zambia
3,106-286,10/2/2024,10/2/2024,Male,At Immunisation Facility,Yes,AA,AA,FA - NO abnormal hemoglobin,zambia
4,106-287,10/3/2024,10/3/2024,Female,At Birthing Facility,Yes,AA,AA,FA - NO abnormal hemoglobin,zambia


In [14]:
df_zam_prelim.columns

Index(['StudyID', 'Date standard POCT test done (D-M-Y)',
       'Date of Standard POCT results (D-M-Y)', 'Sex',
       'When was the newborn screened?', 'Standard POCT done?',
       'Standard POCT Results', ' DBS POCT Results', 'IEF Results', 'country'],
      dtype='object')

In [15]:
def clean_zam_prelim(df):
    df[['Date standard POCT test done (D-M-Y)','Date of Standard POCT results (D-M-Y)']] = df[['Date standard POCT test done (D-M-Y)','Date of Standard POCT results (D-M-Y)']].apply(
        lambda s: pd.to_datetime(s.astype(str).str.strip(), dayfirst=True, errors='coerce')
    )
    df['confirmatory_test'] = 'ief'
    df['Study participant ID'] = 'ZM' + df['StudyID'].astype(str)
    df['Facility ID'] = ''
    df['Age'] = ''
    return df

df_zam_prelim = clean_zam_prelim(df_zam_prelim)

In [16]:
df_zam_prelim.head(2)

,StudyID,Date standard POCT test done (D-M-Y),Date of Standard POCT results (D-M-Y),Sex,When was the newborn screened?,Standard POCT done?,Standard POCT Results,DBS POCT Results,IEF Results,country,confirmatory_test,Study participant ID,Facility ID,Age
0,106-283,2024-04-10,2024-04-10,Female,At Immunisation Facility,Yes,AS,AS,FAS - Sickle cell trait (AS),zambia,ief,ZM106-283,,
1,106-284,2024-04-10,2024-04-10,Female,At Immunisation Facility,Yes,AA,AA,FA - NO abnormal hemoglobin,zambia,ief,ZM106-284,,


In [17]:
df_ng_prelim.head(2)

,record_id,redcap_data_access_group,facility_id_screening,site_name,date_enrolled,dob_402711,age_2,sex,tribe,relationship_of_the_guardi,specify_relationship,marital_status,edu_level,edulevel_spec,guardian_occup,if_other_occupation_please,religion,numb_children,know_scd_2,where_did_you_hear_about_s,others_hear_scd,scd_detected_atbirth_2,is_guardian_scd,heard_nbs,support_screening,patient_in_registry,importance_nbs_results,importance_followup,how_feasible,reasons_no_followup,spec_nofollowup,when_was_the_newborn_scree,dob_402712,sex_2,dbs_collected,standard_poct_done,screening_form_complete,poct_done,date_test_done_poct,date_of_the_results_poct,results_poct,date_test_done_3,date_of_the_results,results_2,date_test_done_hplc,date_of_the_results_hplc,results_hplc,date_test_done_ief,date_of_the_results_ief,results_ief,spec_type_test,date_test_done_4,date_of_the_results_4,results_4,newborn_registryid,reminder_1,reminder_2,reminder_3,subject_comments,poct_and_lab_results_complete,facility_hcw,hcw_studyid,facility_name_hcw,age_hcw,gender_hcw___1,gender_hcw___2,job_title_hcw___1,job_title_hcw___2,job_title_hcw___3,job_title_hcw___4,job_title_hcw___5,job_title_hcw___6,job_title_hcw___7,job_title_hcw___8,spec_hcw_jobtitle,level_of_education,specify_edu,graduation_year,years_experience,know_scd,scd_detected_atbirth,heard_nbs_2,know_nbstests___1,know_nbstests___2,know_nbstests___3,know_nbstests___4,know_nbstests___5,know_nbstests___6,know_nbstests___7,know_nbstests___8,know_nbstests___9,know_nbstests_2___1,know_nbstests_2___2,know_nbstests_2___3,know_nbstests_2___4,know_nbstests_2___5,know_nbstests_2___6,know_nbstests_2___7,know_nbstests_2___8,know_nbstests_2___9,know_nbstests_3___1,know_nbstests_3___2,know_nbstests_3___3,know_nbstests_3___4,know_nbstests_3___5,know_nbstests_3___6,know_nbstests_3___7,know_nbstests_3___8,know_nbstests_3___9,have_you_heard_of_dry_bloo,have_you_received_training,where_do_you_carry_out_dbs___1,where_do_you_carry_out_dbs___2,waitingtime_results,have_you_heard_of_isoelect,have_you_heard_of_high_per,are_you_involved_in_any_ot,if_yes_name_of_the_project,hcw_recommend___1,hcw_recommend___2,hcw_recommend___3,hcw_recommend___4,hcw_recommend___5,dbs_effeciency_hcw___1,dbs_effeciency_hcw___2,dbs_effeciency_hcw___3,dbs_effeciency_hcw___4,dbs_effeciency_hcw___5,healthcare_worker_questionnaire_complete,facility_id,facility_name,job_title_facitily,type_of_facility,country,province,city,number_of_doctors_in_the_f,number_of_nurses_in_the_fa,number_of_pharmacists_in_t,number_of_laboratory_perso,number_of_midwives_in_the,number_of_administrators_i,number_of_medical_health_i,number_of_community_health,number_of_councellors_in_t,number_of_nutritionist_in,types_of_services_availabl___1,types_of_services_availabl___2,types_of_services_availabl___3,types_of_services_availabl___4,types_of_services_availabl___5,if_other_specify_the_type,equipment_for_screening,equipment_for_diagnosis,please_indicate_all_the_te___1,please_indicate_all_the_te___2,please_indicate_all_the_te___3,please_indicate_all_the_te___4,please_indicate_all_the_te___5,please_indicate_all_the_te___6,please_indicate_all_the_te___7,please_indicate_all_the_te___8,please_indicate_all_the_te___9,please_indicate_all_the_te_2___1,please_indicate_all_the_te_2___2,please_indicate_all_the_te_2___3,please_indicate_all_the_te_2___4,please_indicate_all_the_te_2___5,please_indicate_all_the_te_2___6,please_indicate_all_the_te_2___7,please_indicate_all_the_te_2___8,please_indicate_all_the_te_2___9,how_far_much_does_it_cost,distance_hemelectro,cost_hplc,distance_hplc,cost_capillaryelectro,distance_capillaryelectro,cost_ief,distance_ief,cost_pcr,distance_pcr,cost_dnaseque,distance_dnaseque,cost_dbs_poct,distance_dbs_poct,do_you_know_point_of_care,have_you_been_train_in_poi,ease_training_dbspoct,how_long_did_the_training,help_interpret_res,facility_recommend___1,facility_recommend___2,facility_recommend___3,facility_recommend___4,facility_recommend___5,dbs_effeciency_facility___1,dbs_eff

In [18]:
def clean_ng_prelim(df):
    df[['dob_402712','date_enrolled']] = df[['dob_402712','date_enrolled']].apply(
        pd.to_datetime, format='%d %m %Y', errors='coerce'
    )
    df['confirmatory_test'] = 'ief'
    df['Study participant ID'] = 'NG' + df['record_id'].astype(str)
    df['Age_days'] = (df['date_enrolled'] - df['dob_402712']).dt.days
    return df

df_ng_prelim = clean_ng_prelim(df_ng_prelim)

In [19]:
df_zam_prelim.head()

,StudyID,Date standard POCT test done (D-M-Y),Date of Standard POCT results (D-M-Y),Sex,When was the newborn screened?,Standard POCT done?,Standard POCT Results,DBS POCT Results,IEF Results,country,confirmatory_test,Study participant ID,Facility ID,Age
0,106-283,2024-04-10,2024-04-10,Female,At Immunisation Facility,Yes,AS,AS,FAS - Sickle cell trait (AS),zambia,ief,ZM106-283,,
1,106-284,2024-04-10,2024-04-10,Female,At Immunisation Facility,Yes,AA,AA,FA - NO abnormal hemoglobin,zambia,ief,ZM106-284,,
2,106-285,2024-02-10,2024-02-10,Male,At Birthing Facility,Yes,AA,AA,FA - NO abnormal hemoglobin,zambia,ief,ZM106-285,,
3,106-286,2024-02-10,2024-02-10,Male,At Immunisation Facility,Yes,AA,AA,FA - NO abnormal hemoglobin,zambia,ief,ZM106-286,,
4,106-287,2024-03-10,2024-03-10,Female,At Birthing Facility,Yes,AA,AA,FA - NO abnormal hemoglobin,zambia,ief,ZM106-287,,


In [20]:
df_zam_prelim.columns

Index(['StudyID', 'Date standard POCT test done (D-M-Y)',
       'Date of Standard POCT results (D-M-Y)', 'Sex',
       'When was the newborn screened?', 'Standard POCT done?',
       'Standard POCT Results', ' DBS POCT Results', 'IEF Results', 'country',
       'confirmatory_test', 'Study participant ID', 'Facility ID', 'Age'],
      dtype='object')

In [21]:
df_gh_prelim.rename({'record_id':'Study ID', 
                     'Facility ID':'Facility ID',
                     'Study participant ID':'Study participant ID',
                     'Age_days':'Age',
                     'sex_newborn':'Sex',
                     'std_poct_results':'Standard POCT Results',
                     'dbspoct_test_results':'DBS POCT Results',
                     'ief_test_results':'Results',
                     'confirmatory_test': 'confirmatory_test',
                     'country':'country',
                    }, axis=1, inplace=True)

df_ml_prelim.rename({'record_id':'Study ID', 
                     'facility_id_screening':'Facility ID',
                     'participant_id':'Study participant ID',
                     'Age_days':'Age',
                     'sex_newborn':'Sex',
                     'std_poct_results':'Standard POCT Results',
                     'dbspoct_test_results':'DBS POCT Results',
                     'hplc_results':'Results',
                     'confirmatory_test': 'confirmatory_test',
                     'country':'country',
                    }, axis=1, inplace=True)

df_tn_prelim.rename({'record_id':'Study ID', 
                     'facility_id_screening':'Facility ID',
                     'participant_id':'Study participant ID',
                     'Age_days':'Age',
                     'sex_newborn':'Sex',
                     'std_poct_results':'Standard POCT Results',
                     'dbspoct_test_results':'DBS POCT Results',
                     'ief_test_results':'Results',
                     'confirmatory_test': 'confirmatory_test',
                     'country':'country',
                    }, axis=1, inplace=True)

df_ug_prelim.rename({'StudyID':'Study ID', 
                     'Facility ID':'Facility ID',
                     'Study participant ID':'Study participant ID',
                     'Age_days':'Age',
                     'Sex.1':'Sex',
                     'Standard POCT Results':'Standard POCT Results',
                     ' DBS-POCT Results':'DBS POCT Results',
                     'IEF Results':'Results',
                     'confirmatory_test': 'confirmatory_test',
                     'country':'country',
                    }, axis=1, inplace=True)

df_zim_prelim.rename({'StudyID':'Study ID', 
                     'Facility ID':'Facility ID',
                     'Study participant ID':'Study participant ID',
                     'Age':'Age',
                     'Sex':'Sex',
                     'Standard POCT Results':'Standard POCT Results',
                     ' DBS POCT Results':'DBS POCT Results',
                     'IEF Results ':'Results',
                     'confirmatory_test': 'confirmatory_test',
                     'country':'country',
                    }, axis=1, inplace=True)

df_zam_prelim.rename({'StudyID':'Study ID', 
                     'Facility ID':'Facility ID',
                     'Study participant ID':'Study participant ID',
                     'Age':'Age',
                     'Sex':'Sex',
                     'Standard POCT Results':'Standard POCT Results',
                     ' DBS POCT Results':'DBS POCT Results',
                     'IEF Results':'Results',
                     'confirmatory_test': 'confirmatory_test',
                     'country':'country',
                    }, axis=1, inplace=True)

df_ng_prelim.rename({'record_id':'Study ID', 
                     'facility_id_screening':'Facility ID',
                     'Study participant ID':'Study participant ID',
                     'Age_days':'Age',
                     'sex_2':'Sex',
                     'results_poct':'Standard POCT Results',
                     'results_2':'DBS POCT Results',
                     'results_ief':'Results',
                     'confirmatory_test': 'confirmatory_test',
                     'country':'country',
                    }, axis=1, inplace=True)

main_prelim_columns = ['Study ID','Facility ID','Study participant ID','Age', 'Sex',  
                'Standard POCT Results', 'DBS POCT Results', 'Results', 'confirmatory_test', 'country']

df_gh_prelim = df_gh_prelim[main_prelim_columns]
df_ml_prelim = df_ml_prelim[main_prelim_columns]
df_tn_prelim = df_tn_prelim[main_prelim_columns]
df_ug_prelim = df_ug_prelim[main_prelim_columns]
df_zim_prelim = df_zim_prelim[main_prelim_columns]
df_zam_prelim = df_zam_prelim[main_prelim_columns]
df_ng_prelim = df_ng_prelim[main_prelim_columns]

df_all_prelim = pd.concat([df_gh_prelim, df_ml_prelim, df_tn_prelim, 
                           df_ug_prelim, df_zim_prelim, df_zam_prelim, df_ng_prelim], ignore_index=True)

df_all_prelim['country'] = (
    df_all_prelim['country']
    .astype(str)
    .str.strip()
    .str.title()
)

df_all_prelim['Age'] = (
    df_all_prelim['Age']
    .astype(str)
    .str.strip()
    .replace({'': pd.NA, 'nan': pd.NA, 'NaN': pd.NA, 'None': pd.NA})
    .str.extract(r'(-?\d+\.?\d*)', expand=False)   # grab number if present
)

df_all_prelim['Age'] = pd.to_numeric(df_all_prelim['Age'], errors='coerce').round(0).astype('Int64')

df_all_prelim.loc[df_all_prelim['Age'].lt(0), 'Age'] = pd.NA

In [22]:
df_all_prelim.head(2)

,Study ID,Facility ID,Study participant ID,Age,Sex,Standard POCT Results,DBS POCT Results,Results,confirmatory_test,country
0,0001,Kumasi,GH0001,1,Male,AA,AA,FA - NO abnormal hemoglobin,ief,Ghana
1,0002,Kumasi,GH0002,1,Female,AC,AA,FAC - C trait (AC),ief,Ghana


In [23]:
df_all_prelim.shape

(705, 10)

In [24]:
df_all_prelim['country'].value_counts(dropna=False)

country
Mali        104
Ghana       101
Tanzania    100
Uganda      100
Zimbabwe    100
Zambia      100
Nigeria     100
Name: count, dtype: int64

<h1>Quantitative Phase (900)</h1>
<p>Have data frrom all sites...</p>

In [25]:
#Quantitative phase data

df_gh_quant = pd.read_csv('../data/raw/ghana/raw/ghana_phase_2.csv')
df_ml_quant = pd.read_excel('../data/raw/mali/raw/mali_phase_2.xlsx', sheet_name='NewBornScreening_DATA__20')
df_tn_quant = pd.concat([pd.read_csv('../data/raw/tanzania/raw/tanzania_prelim.csv'), pd.read_csv('../data/raw/tanzania/raw/tanzania_phase_2.csv')], ignore_index=True)
df_ug_quant = pd.read_csv('../data/raw/uganda/raw/uganda_phase_2.csv')
df_zim_quant = pd.concat([pd.read_csv('../data/raw/zim_zam/raw/zim.csv'), pd.read_csv('../data/raw/zim_zam/raw/zim_phase_2.csv')], ignore_index=True)
df_zam_quant = pd.concat([pd.read_csv('../data/raw/zim_zam/raw/zam.csv'), pd.read_csv('../data/raw/zim_zam/raw/zam_phase_2.csv')], ignore_index=True)
df_zam_quant = df_zam_quant[df_zam_quant['StudyID'].notna()]
df_ng_quant = pd.read_csv('../data/raw/nigeria/raw/nigeria_phase_2.csv') 
df_ng_prelim = pd.read_csv('../data/raw/nigeria/raw/prelims.csv') 
keep = [c for c in df_ng_quant.columns if c in df_ng_prelim.columns] 
df_ng_quant = pd.concat([df_ng_prelim[keep], df_ng_quant], ignore_index=True)

print('df_gh_quant:', df_gh_quant.shape)
print('df_ml_quant:', df_ml_quant.shape)
print('df_tn_quant:', df_tn_quant.shape)
print('df_ug_quant:', df_ug_quant.shape)
print('df_zim_quant:', df_zim_quant.shape)
print('df_zam_quant:', df_zam_quant.shape)
print('df_ng_quant:', df_ng_quant.shape)
print(' ')
print('quantitative total:', df_gh_quant.shape[0]+df_ml_quant.shape[0]+df_tn_quant.shape[0]+df_ug_quant.shape[0]+df_zim_quant.shape[0]+df_zam_quant.shape[0]+df_ng_quant.shape[0])

df_gh_quant: (999, 89)
df_ml_quant: (1011, 92)
df_tn_quant: (1000, 52)
df_ug_quant: (980, 88)
df_zim_quant: (550, 51)
df_zam_quant: (550, 51)
df_ng_quant: (2507, 82)
 
quantitative total: 7597


In [26]:
df_zim_quant.tail(2)

,StudyID,Facility ID,Study participant ID,Consent date,Date of birth,Age,Sex,When was the newborn screened?,Screening date (D-M-Y),Kind of screening test done (Please check all that apply),Standard POCT done?,Standard POCT Results,DBS POCT Results,IEF Results,Event Name,Date subject signed consent (DD-MM-YY),Date of visit,Date of birth (D-M-Y),Tribe,Relationship of the guardian to the child,If 'Other' please specify the relationship to the child,Marital status,Highest level of education completed,"If 'Other', please specify",Occupation,"If 'Other' occupation, please specify",Religion,Number of children of the mother of the newborn,Sickle cell disease (SCD) can be detected at birth,People with SCD can live long lives,Early detection of SCD is need to provide adequate care for the affected child,It is important that a test for SCD for my child produces the result immediately,I have access to all the information I need on point of care tests for SCD,The result of a point of care test for SCD is available faster than for other screening methods,I would recommend the screening of babies for SCD at other healthcare facilities,I would recommend the use of point of care tests for SCD at other healthcare facilities,"Alongside the point of care test, it is acceptable for a dried blood spot sample to be collected for validation","If someone tests positive for SCD at this healthcare facility it is important that they seek proper medical treatment, either here or at another facility",Do you have any family member with SCD?,"If your baby tests positive, would you like to enrol the baby in SCD registry?","If your baby tests positive for SCD, would you be able to bring your baby back for follow-up?","If 'No', please specify","Do you have any comments, suggestions or questions?",Date of birth (D-M-Y).1,Sex.1,"Of 'Other' screening test, please specify",Staff member collecting sample(s),Complete?,Date DBS-POCT test done (D-M-Y),Date of DBS-POCT results (D-M-Y),DBS-POCT Results
548,607,PTH,NBSZIM551,NaN,NaN,NaN,Female,At Birthing Facility,8/6/2024,DBS PoCT,No,NaN,NaN,NaN,Event 1 (Arm 1: Arm 1_Guardians),8/6/2024,8/6/2024,11/23/1989,Shona,Mother,NaN,Married,Secondary,NaN,Housewife,NaN,Christianity,3.0,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,I don't know,Yes,Yes,NaN,No,8/5/2024,Male,NaN,PG,Complete,8/6/2024,8/6/2024,AA
549,608,PTH,NBSZIM552,NaN,NaN,NaN,Female,At Birthing Facility,8/6/2024,DBS PoCT,No,NaN,NaN,NaN,Event 1 (Arm 1: Arm 1_Guardians),8/6/2024,8/6/2024,12/24/2004,Shona,Mother,NaN,Married,Secondary,NaN,Housewife,NaN,Christianity,1.0,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,I don't know,Yes,Yes,NaN,No,8/5/2024,Male,NaN,PG,Complete,8/6/2024,8/6/2024,AA


In [27]:
df_zam_quant[' DBS-POCT Results'].value_counts(dropna=False)

 DBS-POCT Results
AA     377
NaN    100
AS      70
SS       3
Name: count, dtype: int64

In [28]:
df_zam_quant.rename(columns={'Date subject signed consent (DD-MM-YY)': 'date_enrolled'}, inplace=True)
df_zim_quant.rename(columns={'dbspoct_test_result_date': 'date_enrolled'}, inplace=True)

In [29]:
df_ml_quant.rename(columns={'Date of birth (D-M-Y)': 'dob_guardian', 'Date of birth (D-M-Y).1':'dob_newborn'}, inplace=True)

In [30]:
df_zam_quant.rename(columns={'Date of birth (D-M-Y)': 'dob_guardian', 'Date of birth (D-M-Y).1':'dob_newborn'}, inplace=True)

In [31]:
df_zim_quant.rename(columns={'Date of birth (D-M-Y)': 'dob_guardian', 'Date of birth (D-M-Y).1':'dob_newborn'}, inplace=True)

In [32]:
df_ug_quant.rename(columns={'Date of birth (D-M-Y)': 'dob_newborn', 'Date of birth (D-M-Y).1':'dob_guardian'}, inplace=True)

In [33]:
df_zim_quant.rename(columns={'Date of birth': 'dob_newborn'}, inplace=True)

In [34]:
df_gh_quant['country'] = 'ghana'
df_ml_quant['country'] = 'mali'
df_tn_quant['country'] = 'tanzania'
df_ug_quant['country'] = 'uganda'
df_zim_quant['country'] = 'zimbabwe'
df_zam_quant['country'] = 'zambia'
df_ng_quant['country'] = 'nigeria'

In [35]:
df_gh_quant['participant_id'] = 'GH' + df_gh_quant['record_id'].astype(str)
df_ml_quant['participant_id'] = 'ML' + df_ml_quant['StudyID'].astype(str)
df_tn_quant['participant_id'] = 'TN' + df_tn_quant['record_id'].astype(str)
df_ug_quant['participant_id'] = 'UG' + df_ug_quant['StudyID'].astype(str)
df_zim_quant['participant_id'] = 'ZIM' + df_zim_quant['StudyID'].astype(str)
df_zam_quant['participant_id'] = 'ZAM' + df_zam_quant['StudyID'].astype(str)
df_ng_quant['participant_id'] = 'NG' + df_ng_quant['record_id'].astype(str)

In [36]:
df_zam_quant.head(1)

,StudyID,Date standard POCT test done (D-M-Y),Date of Standard POCT results (D-M-Y),Sex,When was the newborn screened?,Standard POCT done?,Standard POCT Results,DBS POCT Results,IEF Results,Event Name,Facility ID,Study participant ID,date_enrolled,Date of visit,dob_guardian,Tribe,Relationship of the guardian to the child,If 'Other' please specify the relationship to the child,Marital status,Highest level of education completed,"If 'Other', please specify",Occupation,"If 'Other' occupation, please specify",Religion,Number of children of the mother of the newborn,Sickle cell disease (SCD) can be detected at birth,People with SCD can live long lives,Early detection of SCD is need to provide adequate care for the affected child,It is important that a test for SCD for my child produces the result immediately,I have access to all the information I need on point of care tests for SCD,The result of a point of care test for SCD is available faster than for other screening methods,I would recommend the screening of babies for SCD at other healthcare facilities,I would recommend the use of point of care tests for SCD at other healthcare facilities,"Alongside the point of care test, it is acceptable for a dried blood spot sample to be collected for validation","If someone tests positive for SCD at this healthcare facility it is important that they seek proper medical treatment, either here or at another facility",Do you have any family member with SCD?,"If your baby tests positive, would you like to enrol the baby in SCD registry?","If your baby tests positive for SCD, would you be able to bring your baby back for follow-up?","If 'No', please specify","Do you have any comments, suggestions or questions?",Important landmark,dob_newborn,Sex.1,Screening date (D-M-Y),Kind of screening test done (Please check all that apply),"Of 'Other' screening test, please specify",Staff member collecting sample(s),Complete?,Date DBS-POCT test done (D-M-Y),Date of DBS-POCT results (D-M-Y),DBS-POCT Results,country,participant_id
0,106-283,10/4/2024,10/4/2024,Female,At Immunisation Facility,Yes,AS,AS,FAS - Sickle cell trait (AS),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,zambia,ZAM106-283


In [37]:
# df_zim_quant[['Date of birth (D-M-Y).1']]

In [38]:
# df_ug_quant.columns

In [39]:
mapp = {
    'record_id': ['record_id', 'StudyID'],
    'facility_id': ['facility_id_screening', 'Facility ID'],
    'participant_id': ['participant_id', 'Study participant ID'],
    'date_enrolled': ['date_enrolled', 'Date subject signed consent (DD-MM-YY)'],
    'date_of_visit': ['date_of_visit', 'Date of visit', 'Consent date'],
    'dob_guardian': ['dob_guardian', 'Date of birth (D-M-Y).1'],
    'dob_newborn': ['dob_newborn', 'Date of birth (D-M-Y)'],
    'sex_guardian': ['sex_guardian', 'Sex'],
    'sex_newborn': ['sex_newborn', 'Sex.1'],
    'nbs_screening_locale': ['nbs_screening_locale'],
    'tribe_guardian': ['tribe_guardian', 'Tribe'],
    'guardian_relationship': ['guardian_relationship', 'Relationship of the guardian to the child'],
    'specify_relationship': ['specify_relationship', "If 'Other' please specify the relationship to the child"],
    'marital_status_guardian': ['marital_status_guardian', 'Marital status'],
    'edu_level_guardian': ['edu_level_guardian', 'Highest level of education completed'],
    'specify_edu_level_guardian': ['specify_edu_level_guardian', "If 'Other', please specify"],
    'guardian_occupation': ['guardian_occupation', 'Occupation'],
    'alt_guardian_occupation': ['alt_guardian_occupation', "If 'Other' occupation, please specify"],
    'religion': ['religion', 'Religion', 'Religion ', 'religion '],
    'mother_children_count': ['mother_children_count', 'Number of children of the mother of the newborn'],
    'dob_newborn': ['dob_newborn', 'Date of birth (D-M-Y).1'],
    'sex_newborn': ['sex_newborn', 'Sex.1'],
    'age': ['pgage_yrs', 'age', 'Age'],
    'detection_at_birth': ['detection_at_birth', 'Sickle cell disease (SCD) can be detected at birth'],
    'scd_longevity': ['scd_longevity', 'People with SCD can live long lives'],
    'early_detection_importance': ['early_detection_importance', 'Early detection of SCD is need to provide adequate care for the affected child'],
    'need_immediate_result': ['need_immediate_result', 'It is important that a test for SCD for my child produces the result immediately'],
    'enough_poct_info': ['enough_poct_info', 'I have access to all the information I need on point of care tests for SCD'],
    'poct_result_speed': ['poct_result_speed', 'The result of a point of care test for SCD is available faster than for other screening methods'],
    'screening_recommendation': ['screening_recommendation', 'I would recommend the screening of babies for SCD at other healthcare facilities'],
    'poct_recommendation': ['poct_recommendation', 'I would recommend the use of point of care tests for SCD at other healthcare facilities'],
    'dbs_for_validation': ['dbs_for_validation', 'Alongside the point of care test, it is acceptable for a dried blood spot sample to be collected for validation'],
    'proper_treatment': ['proper_treatment', 'If someone tests positive for SCD at this healthcare facility it is important that they seek proper medical treatment, either here or at another facility'],
    'scd_in_the_family': ['scd_in_the_family', 'Do you have any family member with SCD?'],
    'registry_willingness': ['registry_willingness', 'If your baby tests positive, would you like to enrol the baby in SCD registry?'],
    'no_followup_reasons': ['no_followup_reasons', 'If your baby tests positive for SCD, would you be able to bring your baby back for follow-up?'],
    'nofollowup_reason_specified': ['nofollowup_reason_specified', "If 'No', please specify"],
    'screening_date': ['screening_date', 'Screening date (D-M-Y)'],
    'standard_poct': ['kind_of_screening_test___1'],
    'dbs_poct': ['kind_of_screening_test___2'],
    'ief': ['kind_of_screening_test___3'],
    'hplc': ['hplc', 'HPLC'],
    'molecular_testing': ['kind_of_screening_test___5'],
    'other': ['kind_of_screening_test___6'],
    'alt_screening_test': ['alt_screening_test'],
    'screening_test_done': ['Kind of screening test done (Please check all that apply)'],
    'std_poct_done': ['std_poct_done', 'Standard POCT done?'],
    'std_poct_test_date': ['std_poct_test_date', 'Date standard POCT test done (D-M-Y)'],
    'std_poct_test_result_date': ['std_poct_test_result_date', 'Date of Standard POCT results (D-M-Y)'],
    'std_poct_results': ['std_poct_results', 'Standard POCT Results'],
    'dbspoct_test_date': ['dbspoct_test_date', 'Date DBS-POCT test done (D-M-Y)'],
    'dbspoct_test_result_date': ['dbspoct_test_result_date', 'Date of DBS-POCT results (D-M-Y)'],
    'dbspoct_test_results': ['dbspoct_test_results', 'DBS-POCT Results', ' DBS-POCT Results'],
    'ief_test_date': ['ief_test_date', 'Date IEF test done (D-M-Y)'],
    'ief_test_result_date': ['ief_test_result_date', 'Date of IEF results (D-M-Y)'],
    'ief_test_results': ['ief_test_results', 'IEF Results'],
    'hplc_test_date': ['hplc_test_date', 'Date HPLC test done (D-M-Y)'],
    'hplc_result_date': ['hplc_result_date', 'Date of HPLC results (D-M-Y)'],
    'hplc_results': ['hplc_results', 'HPLC Results'],
    'additional_molecular_test': ['additional_molecular_test'],
    'additional_test_date': ['additional_test_date', 'Date additional molecular test done (D-M-Y)'],
    'add_molecular_result_date': ['add_molecular_result_date', 'Date of additional molecular results (D-M-Y)'],
    'add_molecula_test_results': ['add_molecula_test_results', 'Additional molecular test results']
}

In [40]:
def standardize_cols(df, mapp):
    rename = {src: std for std, sources in mapp.items() for src in sources if src in df.columns}
    df = df.rename(columns=rename)
    return df.loc[:, ~df.columns.duplicated()]

In [41]:
df_gh_quant = standardize_cols(df_gh_quant, mapp)
df_ml_quant = standardize_cols(df_ml_quant, mapp)
df_tn_quant = standardize_cols(df_tn_quant, mapp)
df_ug_quant = standardize_cols(df_ug_quant, mapp)
df_zim_quant = standardize_cols(df_zim_quant, mapp)
df_zam_quant = standardize_cols(df_zam_quant, mapp)
df_ng_quant = standardize_cols(df_ng_quant, mapp)

In [42]:
df_zim_quant.head(1)

,record_id,facility_id,participant_id,date_of_visit,dob_newborn,age,sex_guardian,When was the newborn screened?,screening_date,screening_test_done,std_poct_done,std_poct_results,DBS POCT Results,IEF Results,Event Name,date_enrolled,dob_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,"Do you have any comments, suggestions or questions?",sex_newborn,"Of 'Other' screening test, please specify",Staff member collecting sample(s),Complete?,dbspoct_test_date,dbspoct_test_result_date,dbspoct_test_results,country
0,1,PTH,NBSZIM001,1/22/2024,1/21/2024,NaN,Male,At Birthing Facility,1/22/2024,Standard PoCT,Yes,AA,AA,FA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,zimbabwe


In [43]:
df_ug_quant.columns

Index(['record_id', 'Event Name', 'participant_id', 'sex_guardian',
       'dob_newborn', 'date_enrolled', 'Newborn Age (in days)', 'facility_id',
       'Has participant consented?', 'Upload consent form', 'date_of_visit',
       'dob_guardian', 'sex_newborn', 'tribe_guardian',
       'guardian_relationship', 'specify_relationship',
       'marital_status_guardian', 'edu_level_guardian',
       'specify_edu_level_guardian', 'guardian_occupation',
       'alt_guardian_occupation', 'religion', 'mother_children_count',
       'detection_at_birth', 'scd_longevity', 'early_detection_importance',
       'need_immediate_result', 'enough_poct_info', 'poct_result_speed',
       'screening_recommendation', 'poct_recommendation', 'dbs_for_validation',
       'proper_treatment', 'scd_in_the_family', 'registry_willingness',
       'no_followup_reasons', 'nofollowup_reason_specified',
       'Do you have any comments, suggestions or questions?',
       'Street, City, State, ZIP', 'Important landmar

In [44]:
def convert_dates(df, cols, fmt=None):
    for c in cols:
        df[c] = pd.to_datetime(df[c], format=fmt, errors='coerce', dayfirst=True)
        df[c] = df[c].dt.strftime('%d-%m-%Y')
    return df


date_cols = [
    'dob_newborn',
    'date_enrolled',
    'date_of_visit',
    'dob_guardian'
]

religion_map = {
    "MUSULMAN": "Islam",
    "MUSULMANE": "Islam",
    "CHRISTIANISME": "Christianity"
}

df_ml_quant["religion"] = (
    df_ml_quant["religion"]
    .str.strip()
    .str.upper()
    .replace(religion_map)
)

df_gh_quant = convert_dates(df_gh_quant, date_cols, fmt='%d %m %Y')
df_ml_quant = convert_dates(df_ml_quant, date_cols, fmt='%Y-%m-%d')
df_tn_quant = convert_dates(df_tn_quant, date_cols, fmt='%d/%m/%Y')
df_ug_quant = convert_dates(df_ug_quant, date_cols, fmt='%Y-%m-%d')
df_zim_quant = convert_dates(df_zim_quant, date_cols, fmt='%m/%d/%Y')
df_zam_quant = convert_dates(df_zam_quant, date_cols, fmt='%m/%d/%Y')
df_ng_quant = convert_dates(df_ng_quant, ['date_enrolled'], fmt='%m/%d/%Y')

In [45]:
df_zim_quant.columns

Index(['record_id', 'facility_id', 'participant_id', 'date_of_visit',
       'dob_newborn', 'age', 'sex_guardian', 'When was the newborn screened?',
       'screening_date', 'screening_test_done', 'std_poct_done',
       'std_poct_results', ' DBS POCT Results', 'IEF Results ', 'Event Name',
       'date_enrolled', 'dob_guardian', 'tribe_guardian',
       'guardian_relationship', 'specify_relationship',
       'marital_status_guardian', 'edu_level_guardian',
       'specify_edu_level_guardian', 'guardian_occupation',
       'alt_guardian_occupation', 'religion', 'mother_children_count',
       'detection_at_birth', 'scd_longevity', 'early_detection_importance',
       'need_immediate_result', 'enough_poct_info', 'poct_result_speed',
       'screening_recommendation', 'poct_recommendation', 'dbs_for_validation',
       'proper_treatment', 'scd_in_the_family', 'registry_willingness',
       'no_followup_reasons', 'nofollowup_reason_specified',
       'Do you have any comments, suggestions

In [46]:
print('df_gh_quant:', df_gh_quant.shape)
print('df_ml_quant:', df_ml_quant.shape)
print('df_tn_quant:', df_tn_quant.shape)
print('df_ug_quant:', df_ug_quant.shape)
print('df_zim_quant:', df_zim_quant.shape)
print('df_zam_quant:', df_zam_quant.shape)
print('df_ng_quant:', df_ng_quant.shape)

df_gh_quant: (999, 90)
df_ml_quant: (1011, 93)
df_tn_quant: (1000, 53)
df_ug_quant: (980, 89)
df_zim_quant: (550, 50)
df_zam_quant: (550, 52)
df_ng_quant: (2507, 83)


In [47]:
final_quant_columns = [

    # IDENTIFIERS
    'record_id',
    'facility_id',
    'participant_id',
    'country',
    'nbs_screening_locale',

    # ENROLMENTS / VISITS
    'date_enrolled',
    'date_of_visit',
    'screening_date',
    'age',

    # GUARDIAN DEMOGRAPHICS 
    'dob_guardian',
    'sex_guardian',
    'tribe_guardian',
    'guardian_relationship',
    'specify_relationship',
    'marital_status_guardian',
    'edu_level_guardian',
    'specify_edu_level_guardian',
    'guardian_occupation',
    'alt_guardian_occupation',
    'religion',
    'mother_children_count',

    # NEWBORN 
    'dob_newborn',
    'sex_newborn',

    # KNOWLEDGE QUESTIONS 
    'detection_at_birth',
    'scd_longevity',
    'early_detection_importance',
    'need_immediate_result',
    'enough_poct_info',
    'poct_result_speed',
    'screening_recommendation',
    'poct_recommendation',
    'dbs_for_validation',
    'proper_treatment',
    'scd_in_the_family',
    'registry_willingness',
    'no_followup_reasons',
    'nofollowup_reason_specified',

    # TEST RESULTS
    'std_poct_results',
    'dbspoct_test_results',
    'ief_test_results',
    'hplc_results',
    'add_molecula_test_results',
]

In [48]:
dfs = [df_gh_quant, df_ml_quant, df_tn_quant, df_ug_quant, df_zim_quant,df_zam_quant, df_ng_quant]
dfs = [d.reindex(columns=final_quant_columns) for d in dfs]

In [49]:
df_all_quant = pd.concat(dfs, ignore_index=True)

In [50]:
df_all_quant.shape

(7597, 42)

In [51]:
df_all_quant.head(1)

,record_id,facility_id,participant_id,country,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results
0,1,GH001,GH1,ghana,1.0,20-02-2024,20-02-2024,20 02 2024,NaN,12-10-1996,1,Akan,2.0,NaN,4.0,3.0,NaN,5.0,Decorator,Christian,1.0,19-02-2024,0.0,3.0,2.0,4.0,5.0,5.0,4.0,5.0,5.0,4.0,5.0,2.0,1.0,1.0,NaN,1.0,1.0,1.0,NaN,NaN


In [52]:
df_zam_quant['dbspoct_test_results'].value_counts(dropna=False)

dbspoct_test_results
AA     377
NaN    100
AS      70
SS       3
Name: count, dtype: int64

In [53]:
df_all_quant[df_all_quant['country'] == 'Zambia']['dbspoct_test_results'].value_counts(dropna=False)

Series([], Name: count, dtype: int64)

In [54]:
df_all_quant.head()

,record_id,facility_id,participant_id,country,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results
0,1,GH001,GH1,ghana,1.0,20-02-2024,20-02-2024,20 02 2024,NaN,12-10-1996,1,Akan,2.0,NaN,4.0,3.0,NaN,5.0,Decorator,Christian,1.0,19-02-2024,0.0,3.0,2.0,4.0,5.0,5.0,4.0,5.0,5.0,4.0,5.0,2.0,1.0,1.0,NaN,1.0,1.0,1.0,NaN,NaN
1,2,GH001,GH2,ghana,1.0,20-02-2024,20-02-2024,20 02 2024,NaN,NaN,1,Saliga,2.0,NaN,3.0,3.0,NaN,5.0,Seamstress,Muslim,3.0,19-02-2024,1.0,5.0,2.0,4.0,5.0,5.0,3.0,5.0,5.0,5.0,5.0,3.0,1.0,1.0,NaN,3.0,1.0,7.0,NaN,NaN
2,3,GH001,GH3,ghana,1.0,20-02-2024,20-02-2024,20 02 2024,NaN,28-09-1991,1,Frafra,2.0,NaN,3.0,2.0,NaN,5.0,Seamstress,Muslim,2.0,19-02-2024,0.0,3.0,2.0,4.0,5.0,5.0,5.0,5.0,5.0,4.0,5.0,2.0,1.0,1.0,NaN,1.0,1.0,1.0,NaN,NaN
3,4,GH001,GH4,ghana,1.0,20-02-2024,20-02-2024,20 02 2024,26.0,NaN,1,Fulani,2.0,NaN,3.0,4.0,NaN,5.0,Seamstress,Muslim,1.0,09-01-2024,1.0,5.0,4.0,5.0,2.0,5.0,4.0,5.0,5.0,5.0,5.0,2.0,1.0,1.0,NaN,2.0,2.0,6.0,NaN,NaN
4,5,GH001,GH5,ghana,1.0,21-02-2024,21-02-2024,21 02 2024,NaN,01-01-1993,1,Songhai,2.0,NaN,3.0,1.0,NaN,3.0,NaN,Muslim,7.0,21-02-2024,0.0,4.0,3.0,3.0,3.0,4.0,4.0,4.0,4.0,4.0,4.0,2.0,1.0,1.0,NaN,1.0,1.0,1.0,NaN,NaN


In [55]:
df_all_quant['mother_children_count'] = pd.to_numeric(df_all_quant['mother_children_count'],errors='coerce').astype('Int64')

date_cols = ['dob_guardian', 'dob_newborn', 'date_enrolled', 'date_of_visit']

for col in date_cols:
    df_all_quant[col] = pd.to_datetime(df_all_quant[col], dayfirst=True, errors='coerce')

In [56]:
df_all_quant['age_guardian'] = (
    (df_all_quant['date_enrolled'] - df_all_quant['dob_guardian'])
    .dt.days // 365.25
)

df_all_quant.insert(8, 'age_guardian', df_all_quant.pop('age_guardian'))
df_all_quant['age_guardian'] = df_all_quant['age_guardian'].fillna(df_all_quant['age']); df_all_quant.drop(columns='age', inplace=True)
df_all_quant['age_guardian'] = df_all_quant['age_guardian'].astype('Int64')

In [57]:
df_all_quant['age_newborn'] = (
    (df_all_quant['date_enrolled'] - df_all_quant['dob_newborn'])
    .dt.days
)

In [58]:
df_all_quant.head(10)

,record_id,facility_id,participant_id,country,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,ghana,1.0,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,1,Akan,2.0,NaN,4.0,3.0,NaN,5.0,Decorator,Christian,1,2024-02-19,0.0,3.0,2.0,4.0,5.0,5.0,4.0,5.0,5.0,4.0,5.0,2.0,1.0,1.0,NaN,1.0,1.0,1.0,NaN,NaN,1.0
1,2,GH001,GH2,ghana,1.0,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,1,Saliga,2.0,NaN,3.0,3.0,NaN,5.0,Seamstress,Muslim,3,2024-02-19,1.0,5.0,2.0,4.0,5.0,5.0,3.0,5.0,5.0,5.0,5.0,3.0,1.0,1.0,NaN,3.0,1.0,7.0,NaN,NaN,1.0
2,3,GH001,GH3,ghana,1.0,2024-02-20,2024-02-20,20 02 2024,32,1991-09-28,1,Frafra,2.0,NaN,3.0,2.0,NaN,5.0,Seamstress,Muslim,2,2024-02-19,0.0,3.0,2.0,4.0,5.0,5.0,5.0,5.0,5.0,4.0,5.0,2.0,1.0,1.0,NaN,1.0,1.0,1.0,NaN,NaN,1.0
3,4,GH001,GH4,ghana,1.0,2024-02-20,2024-02-20,20 02 2024,26,NaT,1,Fulani,2.0,NaN,3.0,4.0,NaN,5.0,Seamstress,Muslim,1,2024-01-09,1.0,5.0,4.0,5.0,2.0,5.0,4.0,5.0,5.0,5.0,5.0,2.0,1.0,1.0,NaN,2.0,2.0,6.0,NaN,NaN,42.0
4,5,GH001,GH5,ghana,1.0,2024-02-21,2024-02-21,21 02 2024,31,1993-01-01,1,Songhai,2.0,NaN,3.0,1.0,NaN,3.0,NaN,Muslim,7,2024-02-21,0.0,4.0,3.0,3.0,3.0,4.0,4.0,4.0,4.0,4.0,4.0,2.0,1.0,1.0,NaN,1.0,1.0,1.0,NaN,NaN,0.0
5,6,GH001,GH6,ghana,1.0,2024-02-21,2024-02-21,21 02 2024,36,NaT,1,Akan,2.0,NaN,3.0,3.0,NaN,5.0,Food vendor,Christian,4,2024-02-20,1.0,3.0,2.0,5.0,5.0,5.0,4.0,5.0,5.0,5.0,5.0,2.0,1.0,1.0,NaN,1.0,NaN,1.0,NaN,NaN,1.0
6,7,GH001,GH7,ghana,1.0,2024-02-21,2024-02-21,21 02 2024,24,NaT,1,NaN,2.0,NaN,4.0,3.0,NaN,3.0,NaN,Christian,2,2024-02-22,1.0,4.0,2.0,5.0,5.0,5.0,5.0,4.0,5.0,5.0,5.0,2.0,NaN,NaN,NaN,1.0,1.0,1.0,NaN,NaN,-1.0
7,8,GH001,GH8,ghana,1.0,2024-02-21,2024-02-21,21 02 2024,27,1996-03-07,1,Akan,2.0,NaN,4.0,2.0,NaN,5.0,Unemployed,Christian,1,2024-02-15,1.0,3.0,1.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,2.0,1.0,1.0,NaN,1.0,1.0,1.0,NaN,NaN,6.0
8,9,GH001,GH9,ghana,1.0,2024-02-21,2024-02-21,21 02 2024,<NA>,NaT,1,Fante,1.0,NaN,3.0,2.0,NaN,3.0,NaN,Christian,3,2024-02-18,0.0,3.0,2.0,5.0,5.0,5.0,4.0,5.0,5.0,5.0,5.0,2.0,1.0,1.0,NaN,3.0,1.0,7.0,NaN,NaN,3.0
9,10,GH001,GH10,ghana,1.0,2024-02-21,2024-02-21,21 02 2024,33,NaT,1,Bono,2.0,NaN,7.0,3.0,NaN,3.0,NaN,Christian,3,2024-02-20,1.0,2.0,1.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,2.0,1.0,1.0,NaN,1.0,1.0,1.0,NaN,NaN,1.0


In [59]:
df_all_quant['age_newborn'] = df_all_quant['date_of_visit'] - df_all_quant['dob_newborn']

In [60]:
df_all_quant['age_newborn'] = ((df_all_quant['date_of_visit'] - df_all_quant['dob_newborn']).dt.days).astype('Int64')
df_all_quant['age_newborn'] = df_all_quant['age_newborn'].clip(lower=0)
df_all_quant[['age_newborn','date_of_visit','dob_newborn']].head(10)

,age_newborn,date_of_visit,dob_newborn
0,1,2024-02-20,2024-02-19
1,1,2024-02-20,2024-02-19
2,1,2024-02-20,2024-02-19
3,42,2024-02-20,2024-01-09
4,0,2024-02-21,2024-02-21
5,1,2024-02-21,2024-02-20
6,0,2024-02-21,2024-02-22
7,6,2024-02-21,2024-02-15
8,3,2024-02-21,2024-02-18
9,1,2024-02-21,2024-02-20


In [61]:
df_all_quant['sex_guardian'] = df_all_quant['sex_guardian'].replace({1: 'Female', 0: 'Male', 1.0: 'Female', 0.0: 'Male'})

df_all_quant['sex_newborn'] = df_all_quant['sex_newborn'].replace({1: 'Female', 0: 'Male', 1.0: 'Female', 0.0: 'Male'})

df_all_quant['nbs_screening_locale'] = df_all_quant['nbs_screening_locale'].replace({1: 'At Birthing Facility', 2: 'At Immunisation Facility', 1.0: 'At Birthing Facility', 2.0: 'At Immunisation Facility'})

rel_map = {1:'Father',2:'Mother',3:'Aunt',4:'Uncle',5:'Grandparent',6:'Sister',7:'Brother',8:'Cousin',9:'Other'}
df_all_quant['guardian_relationship'] = df_all_quant['guardian_relationship'].replace({**rel_map, **{float(k):v for k,v in rel_map.items()}})

mar_map = {1:'Never been married',2:'Annulled',3:'Married',4:'Cohabiting',5:'Widowed',6:'Divorced',7:'Separated'}
df_all_quant['marital_status_guardian'] = df_all_quant['marital_status_guardian'].replace({**mar_map, **{float(k):v for k,v in mar_map.items()}})

edu_map = {1:'No education',2:'Primary',3:'Secondary',4:'Tertiary',5:'Other'}
df_all_quant['edu_level_guardian'] = df_all_quant['edu_level_guardian'].replace({**edu_map, **{float(k):v for k,v in edu_map.items()}})

occ_map = {1:'Employed',2:'Farmer',3:'Trader',4:'Housewife',5:'Others'}
df_all_quant['guardian_occupation'] = df_all_quant['guardian_occupation'].replace({**occ_map, **{float(k):v for k,v in occ_map.items()}})

df_all_quant['religion'] = (
    df_all_quant['religion'].astype(str).str.strip().str.title()
    .replace({'Christian':'Christianity','Muslim':'Islam','Islamic':'Islam','Christianity':'Christianity'})
)
df_all_quant.loc[df_all_quant['religion'].isin(['Nan','None','']), 'religion'] = pd.NA

In [62]:
df_all_quant.head()

,record_id,facility_id,participant_id,country,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1,2024-02-19,Male,3.0,2.0,4.0,5.0,5.0,4.0,5.0,5.0,4.0,5.0,2.0,1.0,1.0,NaN,1.0,1.0,1.0,NaN,NaN,1
1,2,GH001,GH2,ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3,2024-02-19,Female,5.0,2.0,4.0,5.0,5.0,3.0,5.0,5.0,5.0,5.0,3.0,1.0,1.0,NaN,3.0,1.0,7.0,NaN,NaN,1
2,3,GH001,GH3,ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,32,1991-09-28,Female,Frafra,Mother,NaN,Married,Primary,NaN,Others,Seamstress,Islam,2,2024-02-19,Male,3.0,2.0,4.0,5.0,5.0,5.0,5.0,5.0,4.0,5.0,2.0,1.0,1.0,NaN,1.0,1.0,1.0,NaN,NaN,1
3,4,GH001,GH4,ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,26,NaT,Female,Fulani,Mother,NaN,Married,Tertiary,NaN,Others,Seamstress,Islam,1,2024-01-09,Female,5.0,4.0,5.0,2.0,5.0,4.0,5.0,5.0,5.0,5.0,2.0,1.0,1.0,NaN,2.0,2.0,6.0,NaN,NaN,42
4,5,GH001,GH5,ghana,At Birthing Facility,2024-02-21,2024-02-21,21 02 2024,31,1993-01-01,Female,Songhai,Mother,NaN,Married,No education,NaN,Trader,NaN,Islam,7,2024-02-21,Male,4.0,3.0,3.0,3.0,4.0,4.0,4.0,4.0,4.0,4.0,2.0,1.0,1.0,NaN,1.0,1.0,1.0,NaN,NaN,0


In [63]:
def harmonise_likert(df, columns):
    likert_map = {
        1: 'Strongly Disagree',
        2: 'Disagree',
        3: "Neutral/ I don't know",
        4: 'Agree',
        5: 'Strongly Agree'
    }

    full_map = {**likert_map, **{float(k): v for k, v in likert_map.items()}}

    for col in columns:
        df[col] = df[col].replace(full_map)

    return df

likert_cols = [
    'detection_at_birth',
    'scd_longevity',
    'early_detection_importance',
    'need_immediate_result',
    'enough_poct_info',
    'poct_result_speed',
    'screening_recommendation'
]

df_all_quant = harmonise_likert(df_all_quant, likert_cols)
df_all_quant['screening_recommendation'].value_counts(dropna=False)

screening_recommendation
Agree                    4976
Strongly Agree           1599
Neutral/ I don't know     580
NaN                       332
Disagree                   95
Strongly Disagree          15
Name: count, dtype: int64

In [64]:
def recode(df, col, mapping):
    df[col] = df[col].replace({**mapping, **{float(k): v for k, v in mapping.items() if isinstance(k, int)}})
    return df

likert_map = {1:'Strongly Disagree',2:'Disagree',3:"Neutral/ I don't know",4:'Agree',5:'Strongly Agree'}
scd_map = {1:'Yes',2:'No',3:"I don't know"}
yesno12 = {1:'Yes',2:'No'}
reg_map = {1:'Yes',0:'No','Oui':'Yes','Non':'No'}

for c in ['poct_recommendation','dbs_for_validation','proper_treatment']:
    recode(df_all_quant, c, likert_map)

df_all_quant = recode(df_all_quant, 'scd_in_the_family', scd_map)
df_all_quant = recode(df_all_quant, 'registry_willingness', reg_map)
df_all_quant = recode(df_all_quant, 'no_followup_reasons', yesno12)

In [65]:
def recode_numeric(df, col, mapping):
    df[col] = df[col].replace({**mapping, **{float(k): v for k, v in mapping.items()}})
    return df

std_map = {1:'AA',2:'AS',3:'AC',4:'SS',5:'SC',6:'CC',7:'Indeterminate'}

for c in ['std_poct_results','dbspoct_test_results']:
    recode_numeric(df_all_quant, c, std_map)
    
hemoglobin_pattern_map = {
    1:'FA - NO abnormal hemoglobin',
    2:'AF - NO abnormal hemoglobin',
    3:'FS - Sickle cell disease SS (SCD-SS)/ Sickle cell disease beta-0-thalassemia',
    4:'FSC - Sickle cell disease SC (SCD-SC)',
    5:'FSA - Sickle cell disease beta (+) thalassemia [SCD-beta(+)thal]',
    6:'FAS - Sickle cell trait (AS)',
    7:'FAC - C trait (AC)',
    8:'Other'
}

for c in ['ief_test_results','hplc_results']:
    recode_numeric(df_all_quant, c, hemoglobin_pattern_map)
    
hemoglobin_to_genotype = {
    'FA - NO abnormal hemoglobin': 'AA',
    'AF - NO abnormal hemoglobin': 'AA',
    'FAS - Sickle cell trait (AS)': 'AS',
    'FAC - C trait (AC)': 'AC',
    'FSC - Sickle cell disease SC (SCD-SC)': 'SC',
    'FS - Sickle cell disease SS (SCD-SS)/Sickle cell disease beta-0-thalassemia': 'SS/Sβ0',
    'FS - Sickle cell disease SS (SCD-SS)/ Sickle cell disease beta-0-thalassemia': 'SS/Sβ0',
    'FSA - Sickle cell disease beta (+) thalassemia [SCD-beta(+)thal]': 'Sβ+',
    'Other': 'Other'
}

for c in ['ief_test_results','hplc_results']:
    df_all_quant[c] = df_all_quant[c].replace(hemoglobin_to_genotype)

In [66]:
df_all_quant['religion'] = df_all_quant['religion'].replace({'Moslem': 'Islam'})

In [67]:
df_all_quant['nbs_screening_locale'].value_counts(dropna=False)

nbs_screening_locale
NaN                         3207
At Immunisation Facility    2543
At Birthing Facility        1847
Name: count, dtype: int64

In [68]:
df_all_quant['sex_guardian'].value_counts(dropna=False)

sex_guardian
Female    6768
Male       728
NaN        101
Name: count, dtype: int64

In [69]:
df_all_quant = df_all_quant[df_all_quant['sex_guardian'].notna()]
df_all_quant['sex_guardian'].value_counts(dropna=False)

sex_guardian
Female    6768
Male       728
Name: count, dtype: int64

In [70]:
df_all_quant.shape

(7496, 43)

In [71]:
df_all_quant['country'] = (
    df_all_quant['country']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        'nigeria': 'Nigeria',
        'mali': 'Mali',
        'ghana': 'Ghana',
        'uganda': 'Uganda',
        'tanzania': 'Tanzania',
        'zimbabwe': 'Zimbabwe',
        'zambia': 'Zambia'
    })
)

df_all_quant['country'].value_counts(dropna=False)

country
Nigeria     2407
Mali        1011
Tanzania    1000
Ghana        999
Uganda       979
Zimbabwe     550
Zambia       550
Name: count, dtype: int64

In [72]:
df_all_quant.shape

(7496, 43)

In [73]:
df_all_quant.head()

,record_id,facility_id,participant_id,country,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
1,2,GH001,GH2,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3,2024-02-19,Female,Strongly Agree,Disagree,Agree,Strongly Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,AC,AA,AC,NaN,NaN,1
2,3,GH001,GH3,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,32,1991-09-28,Female,Frafra,Mother,NaN,Married,Primary,NaN,Others,Seamstress,Islam,2,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
3,4,GH001,GH4,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,26,NaT,Female,Fulani,Mother,NaN,Married,Tertiary,NaN,Others,Seamstress,Islam,1,2024-01-09,Female,Strongly Agree,Agree,Strongly Agree,Disagree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,No,Yes,Yes,NaN,AS,AS,AS,NaN,NaN,42
4,5,GH001,GH5,Ghana,At Birthing Facility,2024-02-21,2024-02-21,21 02 2024,31,1993-01-01,Female,Songhai,Mother,NaN,Married,No education,NaN,Trader,NaN,Islam,7,2024-02-21,Male,Agree,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Agree,Agree,Agree,Agree,Agree,Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,0


In [74]:
df_all_quant['ief_test_results'].value_counts(dropna=False)

ief_test_results
NaN       5708
AA        1071
AS         648
SS/Sβ0      44
AC          16
Other        6
SC           3
Name: count, dtype: int64

In [75]:
df_all_quant['sex_guardian'].value_counts(dropna=False)

sex_guardian
Female    6768
Male       728
Name: count, dtype: int64

In [76]:
df_all_quant.shape

(7496, 43)

In [77]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, kruskal

def make_newborn_table(df, country_col="country"):
    desired_order = ["ghana", "mali", "nigeria", "tanzania", "uganda", "zambia", "zimbabwe"]
    label_map = {
        "ghana": "Ghana",
        "mali": "Mali",
        "nigeria": "Nigeria",
        "tanzania": "Tanzania",
        "uganda": "Uganda",
        "zambia": "Zambia",
        "zimbabwe": "Zimbabwe"
    }

    counts = df[country_col].value_counts()
    country_codes = [c for c in desired_order if c in counts.index]
    overall_n = len(df)

    def n_perc(n, d):
        if d == 0:
            return "0 (0.0%)"
        return f"{int(n)} ({100*n/d:.1f}%)"

    overall_col = f"Overall (N={overall_n})\nn(%)"
    country_cols = [(c, f"{label_map.get(c, c.title())} (N={counts.get(c,0)})\nn(%)") for c in country_codes]
    columns = ["Characteristic", overall_col] + [col for _, col in country_cols] + ["p-value"]
    rows = []

    row = dict.fromkeys(columns, "")
    row["Characteristic"] = "Total N (%)"
    row[overall_col] = n_perc(overall_n, overall_n)
    for code, col_name in country_cols:
        row[col_name] = n_perc(counts.get(code, 0), overall_n)
    row["p-value"] = ""
    rows.append(row)

    sex_ct = pd.crosstab(df[country_col], df["sex_newborn"])
    sex_ct = sex_ct.loc[[c for c in country_codes if c in sex_ct.index]]
    sex_p = chi2_contingency(sex_ct)[1]

    row = dict.fromkeys(columns, "")
    row["Characteristic"] = "Sex: Female"
    f_overall = (df["sex_newborn"] == "Female").sum()
    row[overall_col] = n_perc(f_overall, overall_n)
    for code, col_name in country_cols:
        sub = df[df[country_col] == code]
        f = (sub["sex_newborn"] == "Female").sum()
        row[col_name] = n_perc(f, len(sub))
    row["p-value"] = f"{sex_p:.4f}"
    rows.append(row)

    row = dict.fromkeys(columns, "")
    row["Characteristic"] = "Sex: Male"
    m_overall = (df["sex_newborn"] == "Male").sum()
    row[overall_col] = n_perc(m_overall, overall_n)
    for code, col_name in country_cols:
        sub = df[df[country_col] == code]
        m = (sub["sex_newborn"] == "Male").sum()
        row[col_name] = n_perc(m, len(sub))
    row["p-value"] = f"{sex_p:.4f}"
    rows.append(row)

    age_months = df["age_days"] / 30.44
    med = age_months.median()
    q1, q3 = age_months.quantile([0.25, 0.75])

    groups = [df[df[country_col] == c]["age_days"] / 30.44 for c in country_codes]
    kw_p = kruskal(*groups)[1]

    row = dict.fromkeys(columns, "")
    row["Characteristic"] = "Age (months), median (IQR)"
    row[overall_col] = f"{med:.2f} ({q1:.2f}, {q3:.2f})"
    for code, col_name in country_cols:
        sub = df[df[country_col] == code]
        if len(sub) == 0:
            row[col_name] = "n.d."
        else:
            a = sub["age_days"] / 30.44
            med_c = a.median()
            q1_c, q3_c = a.quantile([0.25, 0.75])
            row[col_name] = f"{med_c:.2f} ({q1_c:.2f}, {q3_c:.2f})"
    row["p-value"] = f"{kw_p:.4f}"
    rows.append(row)

    screen_ct = pd.crosstab(df[country_col], df["nbs_screening_locale"])
    screen_ct = screen_ct.loc[[c for c in country_codes if c in screen_ct.index]]
    screen_p = chi2_contingency(screen_ct)[1]

    row = dict.fromkeys(columns, "")
    row["Characteristic"] = "Newborn screening: At birth facility"
    bf_overall = (df["nbs_screening_locale"] == "At Birthing Facility").sum()
    row[overall_col] = n_perc(bf_overall, overall_n)
    for code, col_name in country_cols:
        sub = df[df[country_col] == code]
        bf = (sub["nbs_screening_locale"] == "At Birthing Facility").sum()
        row[col_name] = n_perc(bf, len(sub))
    row["p-value"] = f"{screen_p:.4f}"
    rows.append(row)

    row = dict.fromkeys(columns, "")
    row["Characteristic"] = "Newborn screening: At immunisation"
    imm_overall = (df["nbs_screening_locale"] == "At Immunisation Facility").sum()
    row[overall_col] = n_perc(imm_overall, overall_n)
    for code, col_name in country_cols:
        sub = df[df[country_col] == code]
        imm = (sub["nbs_screening_locale"] == "At Immunisation Facility").sum()
        row[col_name] = n_perc(imm, len(sub))
    row["p-value"] = f"{screen_p:.4f}"
    rows.append(row)

    return pd.DataFrame(rows, columns=columns)

In [78]:
def make_newborn_table(df, country_col="country"):

    desired_order = ["Ghana", "Mali", "Nigeria", "Tanzania", "Uganda", "Zambia", "Zimbabwe"]
    label_map = {
        "Ghana": "Ghana",
        "Mali": "Mali",
        "Nigeria": "Nigeria",
        "Tanzania": "Tanzania",
        "Uganda": "Uganda",
        "Zambia": "Zambia",
        "Zimbabwe": "Zimbabwe"
    }

    counts = df[country_col].value_counts(dropna=False)
    country_codes = [c for c in desired_order if c in counts.index]
    overall_n = len(df)

    def n_perc(n, d):
        if d == 0:
            return "0 (0.0%)"
        return f"{int(n)} ({100*n/d:.1f}%)"

    def safe_chi2_p(ct):
        """
        ct: crosstab dataframe
        Returns p-value or "n.a." if the test is not valid (e.g., empty/1xK).
        """
        if ct is None or ct.size == 0:
            return "n.a."
        if ct.shape[0] < 2 or ct.shape[1] < 2:
            return "n.a."
        try:
            return f"{chi2_contingency(ct)[1]:.4f}"
        except Exception:
            return "n.a."

    overall_col = f"Overall (N={overall_n})\nn(%)"
    country_cols = [(c, f"{label_map.get(c, c.title())} (N={counts.get(c,0)})\nn(%)") for c in country_codes]
    columns = ["Characteristic", overall_col] + [col for _, col in country_cols] + ["p-value"]
    rows = []

    row = dict.fromkeys(columns, "")
    row["Characteristic"] = "Total N (%)"
    row[overall_col] = n_perc(overall_n, overall_n)
    for code, col_name in country_cols:
        row[col_name] = n_perc(counts.get(code, 0), overall_n)
    row["p-value"] = ""
    rows.append(row)

    sex_ct = pd.crosstab(df[country_col], df["sex_newborn"])
    if len(country_codes) > 0 and sex_ct.size > 0:
        sex_ct = sex_ct.loc[[c for c in country_codes if c in sex_ct.index]]
    sex_p = safe_chi2_p(sex_ct)

    row = dict.fromkeys(columns, "")
    row["Characteristic"] = "Sex: Female"
    f_overall = (df["sex_newborn"] == "Female").sum()
    row[overall_col] = n_perc(f_overall, overall_n)
    for code, col_name in country_cols:
        sub = df[df[country_col] == code]
        f = (sub["sex_newborn"] == "Female").sum()
        row[col_name] = n_perc(f, len(sub))
    row["p-value"] = sex_p
    rows.append(row)

    row = dict.fromkeys(columns, "")
    row["Characteristic"] = "Sex: Male"
    m_overall = (df["sex_newborn"] == "Male").sum()
    row[overall_col] = n_perc(m_overall, overall_n)
    for code, col_name in country_cols:
        sub = df[df[country_col] == code]
        m = (sub["sex_newborn"] == "Male").sum()
        row[col_name] = n_perc(m, len(sub))
    row["p-value"] = sex_p
    rows.append(row)

    if "age_newborn" in df.columns:
        age_months_all = (df["age_newborn"] / 30.44).replace([np.inf, -np.inf], np.nan)
        med = age_months_all.median(skipna=True)
        q1, q3 = age_months_all.quantile([0.25, 0.75])

        age_groups = []
        for c in country_codes:
            sub = (df.loc[df[country_col] == c, "age_newborn"] / 30.44).replace([np.inf, -np.inf], np.nan).dropna()
            if len(sub) >= 2 and sub.nunique() > 1:
                age_groups.append(sub)

        if len(age_groups) >= 2:
            try:
                kw_p = kruskal(*age_groups)[1]
                age_p = f"{kw_p:.4f}"
            except Exception:
                age_p = "n.a."
        else:
            age_p = "n.a."

        row = dict.fromkeys(columns, "")
        row["Characteristic"] = "Age (months), median (IQR)"
        if pd.isna(med):
            row[overall_col] = "n.d."
        else:
            row[overall_col] = f"{med:.2f} ({q1:.2f}, {q3:.2f})"

        for code, col_name in country_cols:
            sub = df[df[country_col] == code]
            if len(sub) == 0 or "age_newborn" not in sub.columns:
                row[col_name] = "n.d."
            else:
                a = (sub["age_newborn"] / 30.44).replace([np.inf, -np.inf], np.nan).dropna()
                if len(a) == 0:
                    row[col_name] = "n.d."
                else:
                    med_c = a.median()
                    q1_c, q3_c = a.quantile([0.25, 0.75])
                    row[col_name] = f"{med_c:.2f} ({q1_c:.2f}, {q3_c:.2f})"
        row["p-value"] = age_p
        rows.append(row)
    else:
        row = dict.fromkeys(columns, "")
        row["Characteristic"] = "Age (months), median (IQR)"
        row[overall_col] = "n.d."
        for _, col_name in country_cols:
            row[col_name] = "n.d."
        row["p-value"] = "n.a."
        rows.append(row)

    screen_ct = pd.crosstab(df[country_col], df["nbs_screening_locale"])
    if len(country_codes) > 0 and screen_ct.size > 0:
        screen_ct = screen_ct.loc[[c for c in country_codes if c in screen_ct.index]]
    screen_p = safe_chi2_p(screen_ct)

    row = dict.fromkeys(columns, "")
    row["Characteristic"] = "Newborn screening: At birth facility"
    bf_overall = (df["nbs_screening_locale"] == "At Birthing Facility").sum()
    row[overall_col] = n_perc(bf_overall, overall_n)
    for code, col_name in country_cols:
        sub = df[df[country_col] == code]
        bf = (sub["nbs_screening_locale"] == "At Birthing Facility").sum()
        row[col_name] = n_perc(bf, len(sub))
    row["p-value"] = screen_p
    rows.append(row)

    row = dict.fromkeys(columns, "")
    row["Characteristic"] = "Newborn screening: At immunisation"
    imm_overall = (df["nbs_screening_locale"] == "At Immunisation Facility").sum()
    row[overall_col] = n_perc(imm_overall, overall_n)
    for code, col_name in country_cols:
        sub = df[df[country_col] == code]
        imm = (sub["nbs_screening_locale"] == "At Immunisation Facility").sum()
        row[col_name] = n_perc(imm, len(sub))
    row["p-value"] = screen_p
    rows.append(row)

    return pd.DataFrame(rows, columns=columns)

In [79]:
df_all_quant.shape

(7496, 43)

In [80]:
df_all_quant.head()

,record_id,facility_id,participant_id,country,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
1,2,GH001,GH2,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3,2024-02-19,Female,Strongly Agree,Disagree,Agree,Strongly Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,AC,AA,AC,NaN,NaN,1
2,3,GH001,GH3,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,32,1991-09-28,Female,Frafra,Mother,NaN,Married,Primary,NaN,Others,Seamstress,Islam,2,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
3,4,GH001,GH4,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,26,NaT,Female,Fulani,Mother,NaN,Married,Tertiary,NaN,Others,Seamstress,Islam,1,2024-01-09,Female,Strongly Agree,Agree,Strongly Agree,Disagree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,No,Yes,Yes,NaN,AS,AS,AS,NaN,NaN,42
4,5,GH001,GH5,Ghana,At Birthing Facility,2024-02-21,2024-02-21,21 02 2024,31,1993-01-01,Female,Songhai,Mother,NaN,Married,No education,NaN,Trader,NaN,Islam,7,2024-02-21,Male,Agree,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Agree,Agree,Agree,Agree,Agree,Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,0


In [81]:
table2 = make_newborn_table(df_all_quant)
table2

,Characteristic,Overall (N=7496)\nn(%),Ghana (N=999)\nn(%),Mali (N=1011)\nn(%),Nigeria (N=2407)\nn(%),Tanzania (N=1000)\nn(%),Uganda (N=979)\nn(%),Zambia (N=550)\nn(%),Zimbabwe (N=550)\nn(%),p-value
0,Total N (%),7496 (100.0%),999 (13.3%),1011 (13.5%),2407 (32.1%),1000 (13.3%),979 (13.1%),550 (7.3%),550 (7.3%),
1,Sex: Female,4112 (54.9%),501 (50.2%),523 (51.7%),1221 (50.7%),502 (50.2%),945 (96.5%),219 (39.8%),201 (36.5%),0.0000
2,Sex: Male,3150 (42.0%),486 (48.6%),488 (48.3%),1172 (48.7%),490 (49.0%),34 (3.5%),231 (42.0%),249 (45.3%),0.0000
3,"Age (months), median (IQR)","0.13 (0.00, 1.45)","0.13 (0.03, 1.38)","1.48 (0.33, 2.50)",n.d.,"0.00 (0.00, 0.00)","0.46 (0.03, 1.64)","0.13 (0.03, 0.30)","0.03 (0.03, 0.10)",0.0000
4,Newborn screening: At birth facility,1847 (24.6%),647 (64.8%),0 (0.0%),203 (8.4%),997 (99.7%),0 (0.0%),0 (0.0%),0 (0.0%),0.0000
5,Newborn screening: At immunisation,2543 (33.9%),352 (35.2%),0 (0.0%),2190 (91.0%),1 (0.1%),0 (0.0%),0 (0.0%),0 (0.0%),0.0000


In [82]:
from docx import Document
from docx.shared import Inches

doc = Document()
doc.add_heading('Table 2', level=1)

# Add table
table = doc.add_table(rows=1, cols=len(table2.columns))
table.style = 'Light Shading Accent 1'

hdr_cells = table.rows[0].cells
for i, col_name in enumerate(table2.columns):
    hdr_cells[i].text = str(col_name)

for _, row in table2.iterrows():
    row_cells = table.add_row().cells
    for i, value in enumerate(row):
        row_cells[i].text = str(value)

doc.save("Prevalence_tables/table_2.docx")

In [83]:
df_all_quant.groupby('country')['age_newborn'].agg(
    count='count',
    min='min',
    max='max',
    mean='mean',
    median='median',
    std='std'
).round(2)

,count,min,max,mean,median,std
country,,,,,,
Ghana,990,0,118,22.55,4.0,28.13
Mali,1011,0,1527,53.83,45.0,89.36
Nigeria,0,<NA>,<NA>,<NA>,<NA>,<NA>
Tanzania,997,0,9257,9.45,0.0,293.17
Uganda,976,0,408,30.28,14.0,33.97
Zambia,450,0,8473,30.73,4.0,399.38
Zimbabwe,100,0,72,4.84,1.0,12.66


In [84]:
df_all_quant.head(1)

,record_id,facility_id,participant_id,country,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1


In [85]:
df_all_quant[df_all_quant['country'] == 'Uganda'][['age_newborn', 'date_of_visit', 'date_enrolled', 
                                                   'dob_newborn']].head()

,age_newborn,date_of_visit,date_enrolled,dob_newborn
3010,0,2024-02-23,2024-02-23,2024-02-23
3011,0,2023-10-19,2023-10-19,2023-10-19
3012,1,2023-10-19,2023-10-19,2023-10-18
3013,1,2023-10-20,2023-10-20,2023-10-19
3014,7,2023-10-20,2023-10-20,2023-10-13


In [86]:
df_ug_quantt = pd.read_csv('../data/raw/uganda/raw/uganda_phase_2.csv')
df_ug_quantt.head(1)

,StudyID,Event Name,Study participant ID,Sex,Date of birth (D-M-Y),Date subject signed consent (DD-MM-YY),Newborn Age (in days),Facility ID,Has participant consented?,Upload consent form,Date of visit,Date of birth (D-M-Y).1,Sex.1,Tribe,Relationship of the guardian to the child,If 'Other' please specify the relationship to the child,Marital status,Highest level of education completed,"If 'Other', please specify",Occupation,"If 'Other' occupation, please specify",Religion,Number of children of the mother of the newborn,Sickle cell disease (SCD) can be detected at birth,People with SCD can live long lives,Early detection of SCD is need to provide adequate care for the affected child,It is important that a test for SCD for my child produces the result immediately,I have access to all the information I need on point of care tests for SCD,The result of a point of care test for SCD is available faster than for other screening methods,I would recommend the screening of babies for SCD at other healthcare facilities,I would recommend the use of point of care tests for SCD at other healthcare facilities,"Alongside the point of care test, it is acceptable for a dried blood spot sample to be collected for validation","If someone tests positive for SCD at this healthcare facility it is important that they seek proper medical treatment, either here or at another facility",Do you have any family member with SCD?,"If your baby tests positive, would you like to enrol the baby in SCD registry?","If your baby tests positive for SCD, would you be able to bring your baby back for follow-up?","If 'No', please specify","Do you have any comments, suggestions or questions?","Street, City, State, ZIP",Important landmark,When was the newborn screened?,Screening date (D-M-Y),Kind of screening test done (Please check all that apply),"Of 'Other' screening test, please specify",Staff member collecting sample(s),Complete?,Name and surname of the mother,Standard POCT done?,Date standard POCT test done (D-M-Y),Date of Standard POCT results (D-M-Y),Standard POCT Results,Date DBS-POCT test done (D-M-Y),Date of DBS-POCT results (D-M-Y),DBS-POCT Results,Date HPLC test done (D-M-Y),Date of HPLC results (D-M-Y),HPLC Results,Date IEF test done (D-M-Y),Date of IEF results (D-M-Y),IEF Results,Please specify the type of molecular test that was done,Date additional molecular test done (D-M-Y),Date of additional molecular results (D-M-Y),Additional molecular test results,Study ID in the SCD registry (for the newborn),Reminder 1 (1 month),Method of reminder 1 (choice=SMS),Method of reminder 1 (choice=Phonecall),Method of reminder 1 (choice=In-person visit),Method of reminder 1 (choice=Other),'Other' reminder 1 method,Person responsible for reminder 1,Reminder 2 (2 months),Method of reminder 2 (choice=SMS),Method of reminder 2 (choice=Phonecall),Method of reminder 2 (choice=In-person visit),Method of reminder 2 (choice=Other),'Other' reminder 2 method,Person responsible for reminder 2,Reminder 3 (3 months),Method of reminder 3 (choice=SMS),Method of reminder 3 (choice=Phonecall),Method of reminder 3 (choice=In-person visit),Method of reminder 3 (choice=Other),'Other' reminder 3 method,Person responsible for reminder 3,Comments,Complete?.1
0,890,Event 1 (Arm 1: Arm 1_Guardians),LRRH/NBS/0000,Male,2024-02-23,2024-02-23,29.0,LRRH,Yes,1708690809672.jpg,2024-02-23,1989-07-04,Female,LANGO,Mother,NaN,Married,Secondary,NaN,Others,NaN,CATHOLIC,3,Strongly Agree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,NaN,Yes,Yes,NaN,NONE,KIROMBE NORTH LIRA,School,At Birthing Facility,2024-02-23,DBS PoCT,NaN,NaN,Incomplete,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unchecked,Unchecked,Unchecked,Unchecked,NaN,NaN,NaN,Unchecked,Unchecked,Unchecked,Unchecked,NaN,NaN,NaN,Unchecked,Unchecked,Unchecked,Unchecked,NaN,NaN,NaN,Incomplete


In [87]:
df_all_quant['country'].value_counts(dropna=False)

country
Nigeria     2407
Mali        1011
Tanzania    1000
Ghana        999
Uganda       979
Zimbabwe     550
Zambia       550
Name: count, dtype: int64

In [88]:
print(df_all_quant.shape)
df_all_quant.head()

(7496, 43)


,record_id,facility_id,participant_id,country,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
1,2,GH001,GH2,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3,2024-02-19,Female,Strongly Agree,Disagree,Agree,Strongly Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,AC,AA,AC,NaN,NaN,1
2,3,GH001,GH3,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,32,1991-09-28,Female,Frafra,Mother,NaN,Married,Primary,NaN,Others,Seamstress,Islam,2,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
3,4,GH001,GH4,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,26,NaT,Female,Fulani,Mother,NaN,Married,Tertiary,NaN,Others,Seamstress,Islam,1,2024-01-09,Female,Strongly Agree,Agree,Strongly Agree,Disagree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,No,Yes,Yes,NaN,AS,AS,AS,NaN,NaN,42
4,5,GH001,GH5,Ghana,At Birthing Facility,2024-02-21,2024-02-21,21 02 2024,31,1993-01-01,Female,Songhai,Mother,NaN,Married,No education,NaN,Trader,NaN,Islam,7,2024-02-21,Male,Agree,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Agree,Agree,Agree,Agree,Agree,Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,0


In [89]:
import pandas as pd

def make_facility_table(df, country_col="country", facility_col="facility_id"):
    country_order = ["ghana", "mali", "nigeria", "tanzania", "uganda", "zambia", "zimbabwe"]
    country_label = {
        "Ghana": "Ghana",
        "Mali": "Mali",
        "Nigeria": "Nigeria",
        "Tanzania": "Tanzania",
        "Uganda": "Uganda",
        "Zambia": "Zambia",
        "Zimbabwe": "Zimbabwe",
    }

    N_total = len(df)
    rows = []

    for c in country_order:
        sub = df[df[country_col] == c]
        if sub.empty:
            continue

        n_country = len(sub)
        rows.append({
            "Site facility": country_label.get(c, c.title()),
            "Frequency, n(%)": f"{n_country} ({100*n_country/N_total:.1f}%)"
        })

        fac_counts = (
            sub[facility_col]
            .value_counts()
            .sort_index()
        )

        for fac, n in fac_counts.items():
            rows.append({
                "Site facility": f"  {fac}",
                "Frequency, n(%)": f"{n} ({100*n/N_total:.1f}%)"
            })

    return pd.DataFrame(rows, columns=["Site facility", "Frequency, n(%)"])

In [90]:
df_all_quant.groupby('country')['facility_id'].unique()

country
Ghana            [GH001,  GH001, GH002, MR313, G, nan, MP115]
Mali        [CSCOM YIRIMADIO, CSCOMYIRIMADIO, CSCOM YIRMAD...
Nigeria     [PHC AZUIYIOKWU, AKTH, BDTH/BDW, BDTH/URM, ASA...
Tanzania                                    [BMC-TZ, BMC-TZ ]
Uganda      [LRRH, LRRA, LRR, LRRH ,  LRRH, LR4H, LRTH, Lr...
Zambia              [nan, UTH, MFLH, MFLH , UTH , Uth,  UTH ]
Zimbabwe                               [PTH, MTH,  MTH, MTH ]
Name: facility_id, dtype: object

In [91]:
df_all_quant.groupby('country')['facility_id'].nunique()

country
Ghana         6
Mali          3
Nigeria     111
Tanzania      2
Uganda       11
Zambia        6
Zimbabwe      4
Name: facility_id, dtype: int64

In [92]:
def make_guardian_table(df: pd.DataFrame) -> pd.DataFrame:
    import numpy as np
    import pandas as pd
    from scipy.stats import chi2_contingency

    groups = {
        "Sex": ("sex_guardian", [("Female", "Female"), ("Male", "Male")]),
        "Marital status": (
            "marital_status_guardian",
            [
                ("Married", "Married"),
                ("Never been married", "Never been married"),
                ("Separated", "Separated"),
                ("Divorced", "Divorced"),
                ("Widowed", "Widowed"),
                ("Cohabiting", "Cohabiting"),
                ("Annulled", "Annulled"),
            ],
        ),
        "Highest level of education": (
            "edu_level_guardian",
            [
                ("No education", "No education"),
                ("Primary", "Primary"),
                ("Secondary", "Secondary"),
                ("Tertiary", "Tertiary"),
                ("Other", "Other"),
            ],
        ),
        "Occupation": (
            "guardian_occupation",
            [
                ("Employed", "Employed"),
                ("Farmer", "Farmer"),
                ("Trader", "Trader"),
                ("Housewife", "Housewife"),
                ("Others", "Others"),
            ],
        ),
        "Number of children": (
            "mother_children_count",
            [(1, "1"), (2, "2"), (3, "3"), (4, "4"), (5, "5"), (6, "6"), (7, "7")],
        ),
        "Religion": (
            "religion",
            [("Christianity", "Christian"), ("Islam", "Muslim"), ("Other", "Other")],
        ),
    }

    country_order = ["Ghana", "Mali", "Nigeria", "Tanzania", "Uganda", "Zambia", "Zimbabwe"]
    countries = [c for c in country_order if c in df["country"].unique()]

    rows = []

    for section_label, (col, levels) in groups.items():
        p_section = "—"
        try:
            sub = df[df["country"].isin(countries) & df[col].notna()][["country", col]]
            if not sub.empty:
                level_vals = [v for v, _ in levels]
                ct = pd.crosstab(sub["country"], sub[col]).reindex(index=countries, columns=level_vals, fill_value=0)
                ct2 = ct.loc[ct.sum(axis=1).gt(0), ct.sum(axis=0).gt(0)]  # drop empty rows/cols
                if ct2.shape[0] >= 2 and ct2.shape[1] >= 2:
                    chi2, p, dof, expected = chi2_contingency(ct2.values)
                    if not (expected == 0).any():
                        p_section = f"{p:.3f}"
        except:
            p_section = "—"

        for i, (val, label) in enumerate(levels):
            char_label = f"{section_label}\n{label}" if i == 0 else label
            row = {"Characteristic": char_label}

            denom_overall = df[col].notna().sum()
            n_overall = (df[col] == val).sum()
            pct_overall = (n_overall / denom_overall * 100) if denom_overall else 0
            row["Overall"] = f"{n_overall} ({pct_overall:.1f}%)"

            for c in countries:
                subc = df[df["country"] == c]
                d = subc[col].notna().sum()
                n = (subc[col] == val).sum()
                pct = (n / d * 100) if d else 0
                row[c.capitalize()] = f"{n} ({pct:.1f}%)"

            row["p-value"] = p_section
            rows.append(row)

    out = pd.DataFrame(rows)
    out = out[["Characteristic", "Overall"] + [c.capitalize() for c in countries] + ["p-value"]]
    return out

In [93]:
def make_guardian_table(df: pd.DataFrame) -> pd.DataFrame:

    groups = {
        "Sex": ("sex_guardian", [("Female", "Female"), ("Male", "Male")]),
        "Marital status": (
            "marital_status_guardian",
            [
                ("Married", "Married"),
                ("Never been married", "Never been married"),
                ("Separated", "Separated"),
                ("Divorced", "Divorced"),
                ("Widowed", "Widowed"),
                ("Cohabiting", "Cohabiting"),
                ("Annulled", "Annulled"),
            ],
        ),
        "Highest level of education": (
            "edu_level_guardian",
            [
                ("No education", "No education"),
                ("Primary", "Primary"),
                ("Secondary", "Secondary"),
                ("Tertiary", "Tertiary"),
                ("Other", "Other"),
            ],
        ),
        "Occupation": (
            "guardian_occupation",
            [
                ("Employed", "Employed"),
                ("Farmer", "Farmer"),
                ("Trader", "Trader"),
                ("Housewife", "Housewife"),
                ("Others", "Others"),
            ],
        ),
        "Number of children": (
            "mother_children_count",
            [(1, "1"), (2, "2"), (3, "3"), (4, "4"), (5, "5"), (6, "6"), (7, "7")],
        ),
        "Religion": (
            "religion",
            [("Christianity", "Christian"), ("Islam", "Muslim"), ("Other", "Other")],
        ),
    }

    country_order = ["Ghana", "Mali", "Nigeria", "Tanzania", "Uganda", "Zambia", "Zimbabwe"]
    countries = [c for c in country_order if c in df["country"].unique()]

    rows = []

    anchor_col = "sex_guardian"
    denom_overall = df[anchor_col].notna().sum()
    total_row = {"Characteristic": "Total (N)", "Overall": f"{denom_overall}"}
    for c in countries:
        denom_c = df.loc[df["country"].eq(c), anchor_col].notna().sum()
        total_row[c.capitalize()] = f"{denom_c}"
    total_row["p-value"] = "—"
    rows.append(total_row)

    for section_label, (col, levels) in groups.items():
        p_section = "—"
        try:
            sub = df[df["country"].isin(countries) & df[col].notna()][["country", col]]
            if not sub.empty:
                level_vals = [v for v, _ in levels]
                ct = pd.crosstab(sub["country"], sub[col]).reindex(index=countries, columns=level_vals, fill_value=0)
                ct2 = ct.loc[ct.sum(axis=1).gt(0), ct.sum(axis=0).gt(0)]
                if ct2.shape[0] >= 2 and ct2.shape[1] >= 2:
                    chi2, p, dof, expected = chi2_contingency(ct2.values)
                    if not (expected == 0).any():
                        p_section = f"{p:.3f}"
        except:
            p_section = "—"

        for i, (val, label) in enumerate(levels):
            char_label = f"{section_label}\n{label}" if i == 0 else label
            row = {"Characteristic": char_label}

            denom_overall = df[col].notna().sum()
            n_overall = (df[col] == val).sum()
            pct_overall = (n_overall / denom_overall * 100) if denom_overall else 0
            row["Overall"] = f"{n_overall} ({pct_overall:.1f}%)"

            for c in countries:
                subc = df[df["country"] == c]
                d = subc[col].notna().sum()
                n = (subc[col] == val).sum()
                pct = (n / d * 100) if d else 0
                row[c.capitalize()] = f"{n} ({pct:.1f}%)"

            row["p-value"] = p_section
            rows.append(row)

    out = pd.DataFrame(rows)
    out = out[["Characteristic", "Overall"] + [c.capitalize() for c in countries] + ["p-value"]]
    return out

In [94]:
df_all_quant['mother_children_count'] = (
    df_all_quant['mother_children_count']
        .astype(str)
        .str.replace(r'[^0-9.]', '', regex=True)  # keep only digits
        .replace('', np.nan)                      # empty → NaN
        .astype(float)
)

In [95]:
df_all_quant.shape

(7496, 43)

In [96]:
df_all_quant['sex_guardian'].value_counts(dropna=False)

sex_guardian
Female    6768
Male       728
Name: count, dtype: int64

In [97]:
7597-101

7496

In [98]:
6768+728

7496

In [99]:
df_all_quant[df_all_quant['sex_guardian'].isna() == True]

,record_id,facility_id,participant_id,country,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn


In [100]:
df_all_quant['country'].value_counts(dropna=False)

country
Nigeria     2407
Mali        1011
Tanzania    1000
Ghana        999
Uganda       979
Zimbabwe     550
Zambia       550
Name: count, dtype: int64

In [101]:
df_all_quant[df_all_quant['sex_guardian'].isna() == True]

,record_id,facility_id,participant_id,country,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn


In [102]:
df_all_quant[df_all_quant['country'] == 'Zambia'].shape

(550, 43)

In [103]:
df_all_quant[df_all_quant['country'] == 'Zambia'].tail()

,record_id,facility_id,participant_id,country,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
5085,740,UTH,NBSZAM312,Zambia,NaN,2024-09-10,2024-09-10,9/10/2024,34,1990-05-20,Female,Chewe,Mother,NaN,Married,Primary,NaN,Housewife,NaN,Christianity,3.0,2024-09-09,Male,Neutral/ I don't know,Agree,Strongly Agree,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Strongly Agree,Agree,Neutral/ I don't know,Agree,No,Yes,Yes,NaN,NaN,AA,NaN,NaN,NaN,1
5086,741,UTH,NBSZAM313,Zambia,NaN,2024-09-10,2024-09-10,9/10/2024,38,1986-02-15,Female,Tonga,Mother,NaN,Married,Primary,NaN,Housewife,NaN,Christianity,4.0,2024-09-09,Female,Neutral/ I don't know,Agree,Agree,Agree,Neutral/ I don't know,Neutral/ I don't know,Agree,Neutral/ I don't know,Neutral/ I don't know,Agree,No,Yes,Yes,NaN,NaN,AA,NaN,NaN,NaN,1
5087,742,UTH,NBSZAM314,Zambia,NaN,2024-09-11,2024-09-11,9/11/2024,24,2000-01-03,Female,Kaonda,Mother,NaN,Married,Primary,NaN,Housewife,NaN,Christianity,2.0,2024-09-09,Male,Neutral/ I don't know,Agree,Agree,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Agree,Agree,Neutral/ I don't know,Agree,No,Yes,Yes,NaN,NaN,AA,NaN,NaN,NaN,2
5088,743,UTH,NBSZAM315,Zambia,NaN,2024-09-11,2024-09-11,9/11/2024,33,1991-02-14,Female,Bemba,Mother,NaN,Married,Primary,NaN,Housewife,NaN,Christianity,3.0,2024-09-10,Female,Neutral/ I don't know,Agree,Agree,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,No,Yes,Yes,NaN,NaN,AA,NaN,NaN,NaN,1
5089,744,UTH,NBSZAM316,Zambia,NaN,2024-09-11,2024-09-11,9/11/2024,29,1995-05-02,Female,Lozi,Mother,NaN,Married,Primary,NaN,Housewife,NaN,Christianity,1.0,2024-09-10,Female,Neutral/ I don't know,Agree,Agree,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Agree,Agree,Neutral/ I don't know,Agree,No,Yes,Yes,NaN,NaN,AA,NaN,NaN,NaN,1


In [104]:
df_zam_quant1 = pd.read_csv('../data/raw/zim_zam/raw/zam_phase_2.csv')
df_zam_quant1.shape

(550, 47)

In [105]:
df_zam_quant1.tail()

,StudyID,Event Name,Facility ID,Study participant ID,Date subject signed consent (DD-MM-YY),Date of visit,Date of birth (D-M-Y),Sex,Tribe,Relationship of the guardian to the child,If 'Other' please specify the relationship to the child,Marital status,Highest level of education completed,"If 'Other', please specify",Occupation,"If 'Other' occupation, please specify",Religion,Number of children of the mother of the newborn,Sickle cell disease (SCD) can be detected at birth,People with SCD can live long lives,Early detection of SCD is need to provide adequate care for the affected child,It is important that a test for SCD for my child produces the result immediately,I have access to all the information I need on point of care tests for SCD,The result of a point of care test for SCD is available faster than for other screening methods,I would recommend the screening of babies for SCD at other healthcare facilities,I would recommend the use of point of care tests for SCD at other healthcare facilities,"Alongside the point of care test, it is acceptable for a dried blood spot sample to be collected for validation","If someone tests positive for SCD at this healthcare facility it is important that they seek proper medical treatment, either here or at another facility",Do you have any family member with SCD?,"If your baby tests positive, would you like to enrol the baby in SCD registry?","If your baby tests positive for SCD, would you be able to bring your baby back for follow-up?","If 'No', please specify","Do you have any comments, suggestions or questions?",Important landmark,Date of birth (D-M-Y).1,Sex.1,When was the newborn screened?,Screening date (D-M-Y),Kind of screening test done (Please check all that apply),"Of 'Other' screening test, please specify",Staff member collecting sample(s),Complete?,Standard POCT Results,Standard POCT done?,Date DBS-POCT test done (D-M-Y),Date of DBS-POCT results (D-M-Y),DBS-POCT Results
545,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
546,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
547,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
548,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
549,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [106]:
df_zam_quant1['StudyID'].isna().sum()

np.int64(100)

In [107]:
df_zam_quant1[' DBS-POCT Results'].isna().sum()

np.int64(100)

In [108]:
df_all_quant['country'].value_counts(dropna=False)

country
Nigeria     2407
Mali        1011
Tanzania    1000
Ghana        999
Uganda       979
Zimbabwe     550
Zambia       550
Name: count, dtype: int64

In [109]:
df_all_quant.head(2)

,record_id,facility_id,participant_id,country,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
1,2,GH001,GH2,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3.0,2024-02-19,Female,Strongly Agree,Disagree,Agree,Strongly Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,AC,AA,AC,NaN,NaN,1


In [110]:
df_all_quant[df_all_quant['country'] == 'Ghana']['dbspoct_test_results'].value_counts(dropna=False)

dbspoct_test_results
AA               801
NaN               94
AS                81
Indeterminate      7
AC                 6
SS                 5
SC                 3
CC                 2
Name: count, dtype: int64

In [111]:
df_all_quant[df_all_quant['country'] == 'Mali']['dbspoct_test_results'].value_counts(dropna=False)

dbspoct_test_results
AA    802
AS     99
AC     97
SC      6
CC      5
SS      2
Name: count, dtype: int64

In [112]:
df_all_quant[df_all_quant['country'] == 'Zimbabwe']['dbspoct_test_results'].value_counts(dropna=False)

dbspoct_test_results
AA     420
NaN    100
AS      30
Name: count, dtype: int64

In [113]:
df_all_quant[df_all_quant['country'] == 'Tanzania']['dbspoct_test_results'].value_counts(dropna=False)

dbspoct_test_results
AA     820
AS     163
SS      14
NaN      3
Name: count, dtype: int64

In [114]:
df_all_quant[df_all_quant['country'] == 'Uganda']['dbspoct_test_results'].value_counts(dropna=False)

dbspoct_test_results
AA               745
AS               150
NaN               73
SS                 9
Indeterminate      2
Name: count, dtype: int64

In [115]:
df_all_quant[df_all_quant['country'] == 'Zambia']['dbspoct_test_results'].value_counts(dropna=False)

dbspoct_test_results
AA     377
NaN    100
AS      70
SS       3
Name: count, dtype: int64

In [116]:
df_zam_quant1 = pd.read_csv('../data/raw/zim_zam/raw/zam_phase_2.csv')
df_zam_quant1.shape

(550, 47)

In [117]:
df_zam_quant1.head(1)

,StudyID,Event Name,Facility ID,Study participant ID,Date subject signed consent (DD-MM-YY),Date of visit,Date of birth (D-M-Y),Sex,Tribe,Relationship of the guardian to the child,If 'Other' please specify the relationship to the child,Marital status,Highest level of education completed,"If 'Other', please specify",Occupation,"If 'Other' occupation, please specify",Religion,Number of children of the mother of the newborn,Sickle cell disease (SCD) can be detected at birth,People with SCD can live long lives,Early detection of SCD is need to provide adequate care for the affected child,It is important that a test for SCD for my child produces the result immediately,I have access to all the information I need on point of care tests for SCD,The result of a point of care test for SCD is available faster than for other screening methods,I would recommend the screening of babies for SCD at other healthcare facilities,I would recommend the use of point of care tests for SCD at other healthcare facilities,"Alongside the point of care test, it is acceptable for a dried blood spot sample to be collected for validation","If someone tests positive for SCD at this healthcare facility it is important that they seek proper medical treatment, either here or at another facility",Do you have any family member with SCD?,"If your baby tests positive, would you like to enrol the baby in SCD registry?","If your baby tests positive for SCD, would you be able to bring your baby back for follow-up?","If 'No', please specify","Do you have any comments, suggestions or questions?",Important landmark,Date of birth (D-M-Y).1,Sex.1,When was the newborn screened?,Screening date (D-M-Y),Kind of screening test done (Please check all that apply),"Of 'Other' screening test, please specify",Staff member collecting sample(s),Complete?,Standard POCT Results,Standard POCT done?,Date DBS-POCT test done (D-M-Y),Date of DBS-POCT results (D-M-Y),DBS-POCT Results
0,106-1,Event 1 (Arm 1: Arm 1_Guardians),UTH,NBSZAM020,7/7/2024,7/7/2024,6/7/1986,Female,Chewe,Mother,NaN,Married,Secondary,NaN,Housewife,NaN,Christian - Pentecostal,5,Agree,Agree,Strongly Agree,Strongly Agree,Agree,Neutral/ I don't know,Strongly Agree,Agree,Neutral/ I don't know,Strongly Agree,Yes,Yes,Yes,NaN,To introduce sickle cell testing to all local ...,Musha milling,6/15/2024,Female,At Birthing Facility,7/7/2024,DBS PoCT,NaN,Martha Mwale,Complete,NaN,No,7/7/2024,7/10/2024,AS


In [118]:
df_zam_quant1[' DBS-POCT Results'].value_counts(dropna=False)

 DBS-POCT Results
AA     377
NaN    100
AS      70
SS       3
Name: count, dtype: int64

In [119]:
guardian_table = make_guardian_table(df_all_quant)
guardian_table

,Characteristic,Overall,Ghana,Mali,Nigeria,Tanzania,Uganda,Zambia,Zimbabwe,p-value
0,Total (N),7496,999,1011,2407,1000,979,550,550,—
1,Sex\nFemale,6768 (90.3%),985 (98.6%),1007 (99.6%),2308 (95.9%),999 (99.9%),489 (49.9%),487 (88.5%),493 (89.6%),0.000
2,Male,728 (9.7%),14 (1.4%),4 (0.4%),99 (4.1%),1 (0.1%),490 (50.1%),63 (11.5%),57 (10.4%),0.000
3,Marital status\nMarried,6735 (93.1%),766 (76.9%),1002 (99.1%),2298 (97.8%),911 (91.2%),926 (94.6%),392 (87.1%),440 (97.8%),0.000
4,Never been married,247 (3.4%),90 (9.0%),6 (0.6%),34 (1.4%),49 (4.9%),9 (0.9%),54 (12.0%),5 (1.1%),0.000
5,Separated,19 (0.3%),8 (0.8%),0 (0.0%),2 (0.1%),0 (0.0%),5 (0.5%),0 (0.0%),4 (0.9%),0.000
6,Divorced,15 (0.2%),2 (0.2%),1 (0.1%),2 (0.1%),2 (0.2%),6 (0.6%),1 (0.2%),1 (0.2%),0.000
7,Widowed,6 (0.1%),0 (0.0%),0 (0.0%),3 (0.1%),0 (0.0%),1 (0.1%),2 (0.4%),0 (0.0%),0.000
8,Cohabiting,172 (2.4%),99 (9.9%),0 (0.0%),7 (0.3%),34 (3.4%),32 (3.3%),0 (0.0%),0 (0.0%),0.000
9,Annulled,40 (0.6%),31 (3.1%),2 (0.2%),3 (0.1%),3 (0.3%),0 (0.0%),1 (0.2%),0 (0.0%),0.000


In [121]:
from docx import Document
from docx.shared import Inches

doc = Document()
doc.add_heading('Table 1', level=1)

# Add table
table = doc.add_table(rows=1, cols=len(guardian_table.columns))
table.style = 'Light Shading Accent 1'

hdr_cells = table.rows[0].cells
for i, col_name in enumerate(guardian_table.columns):
    hdr_cells[i].text = str(col_name)

for _, row in guardian_table.iterrows():
    row_cells = table.add_row().cells
    for i, value in enumerate(row):
        row_cells[i].text = str(value)

doc.save("Prevalence_tables/table_1.docx")

In [122]:
df_all_quant.shape

(7496, 43)

In [123]:
df_all_quant['religion'].value_counts(dropna=False)

religion
Christianity                           2916
Islam                                  2392
<NA>                                   1223
Catholic                                356
Anglican                                272
Pag                                     139
Protestant                               46
Angilican                                46
Sda                                      30
Angillican                                5
Born Again                                5
Cathoilc                                  4
Pentacostal                               4
Yahweh                                    4
Pga                                       4
P A G                                     4
Christian (Jw)                            3
Pentecostal                               3
Christian Jw                              3
Traditional Religion                      2
Pentacostal Church                        2
Lango                                     2
Mulokole               

In [124]:
df_all_quant.head(2)

,record_id,facility_id,participant_id,country,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
1,2,GH001,GH2,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3.0,2024-02-19,Female,Strongly Agree,Disagree,Agree,Strongly Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,AC,AA,AC,NaN,NaN,1


In [125]:
df_all_quant['record_id'].isna().sum()

np.int64(0)

In [126]:
df_all_quant['religion'].value_counts(dropna=False)
pd.crosstab(df_all_quant['country'], df_all_quant['religion'])

religion,Aanglicn,Adventist,Anganlic,Angelican,Angilican,Angillican,Anglica,Anglican,Anglican2,Born Again,Cagholic,Cathoilc,"Cathol,Ic",Catholi,Catholic,Catholiv,Christian (Jw),Christian (New Apostolic),Christian (Ucz),Christian - Pentecostal,Christian - United Church Of Zambia,Christian 01,Christian Jw,Christianity,Christianity 3,Cstholic,Elim,Islam,Jehova Witness,Lango,Ls,Mulism,Mulokole,P A G,P A G3,P C U,Pag,Pentacostal,Pentacostal Church,Pentecostal,Pentocostal,Pga,Protesant,Protestant,Sad,Sda,Seven Day Adventist,Shona,Traditional Religion,Yahweh,Yaweh,Yaweh Pentocoast
country,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Ghana,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,553,0,0,0,442,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Mali,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,16,0,0,0,993,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Nigeria,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1465,0,0,0,928,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0
Uganda,1,1,1,1,46,5,1,272,1,5,1,4,1,1,356,1,0,0,0,0,0,0,0,0,0,1,1,26,1,2,1,1,1,4,1,1,139,4,2,3,1,4,1,46,1,30,1,0,0,4,1,1
Zambia,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,1,1,1,1,1,3,436,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Zimbabwe,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,446,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0


In [127]:
df_all_quant.head()

,record_id,facility_id,participant_id,country,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
1,2,GH001,GH2,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3.0,2024-02-19,Female,Strongly Agree,Disagree,Agree,Strongly Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,AC,AA,AC,NaN,NaN,1
2,3,GH001,GH3,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,32,1991-09-28,Female,Frafra,Mother,NaN,Married,Primary,NaN,Others,Seamstress,Islam,2.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
3,4,GH001,GH4,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,26,NaT,Female,Fulani,Mother,NaN,Married,Tertiary,NaN,Others,Seamstress,Islam,1.0,2024-01-09,Female,Strongly Agree,Agree,Strongly Agree,Disagree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,No,Yes,Yes,NaN,AS,AS,AS,NaN,NaN,42
4,5,GH001,GH5,Ghana,At Birthing Facility,2024-02-21,2024-02-21,21 02 2024,31,1993-01-01,Female,Songhai,Mother,NaN,Married,No education,NaN,Trader,NaN,Islam,7.0,2024-02-21,Male,Agree,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Agree,Agree,Agree,Agree,Agree,Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,0


In [128]:
df_all_quant.shape

(7496, 43)

In [129]:
df_all_quant['std_poct_results'].value_counts(dropna=False)

std_poct_results
NaN              6394
AA                912
AS                114
AC                 37
CC                 18
SS                  9
Indeterminate       8
SC                  4
Name: count, dtype: int64

<h1>Preliminary</h1>

In [130]:
def make_poct_prevalence_table(df):
    country_order = ["Ghana", "Mali", "Nigeria", "Tanzania", "Uganda", "Zambia", "Zimbabwe"]

    categories = {
        "SCD": ["SS", "SC"],
        "SCT": ["AS"],
        "Normal": ["AA"],
        "Other Hb types": ["AC", "CC"]
    }

    rows = []

    contingency = {cat: [] for cat in categories.keys()}

    overall_total = len(df)

    for country in country_order:
        sub = df[df["country"] == country]
        N = len(sub)

        if N == 0:
            continue

        out = {"Site": country.capitalize()}
        out["Total, n(%)"] = f"{N} ({100*N/overall_total:.1f}%)"

        for cat_label, cat_vals in categories.items():
            n_cat = sub["Standard POCT Results"].isin(cat_vals).sum()
            pct = 100 * n_cat / N if N > 0 else 0
            out[f"{cat_label} n(%)"] = f"{n_cat} ({pct:.1f}%)"
            contingency[cat_label].append(n_cat)

        rows.append(out)

    obs = []
    for cat in categories.keys():
        obs.append(contingency[cat])
    obs = np.array(obs).T  

    cont = pd.crosstab(df_all['country'], df_all['Standard POCT Results']).loc[lambda x: x.sum(1).gt(0), lambda x: x.sum(0).gt(0)]
    chi2, p_value, dof, expected = chi2_contingency(cont.values)

    final_rows = []
    for r in rows:
        r["p-value"] = f"{p_value:.4f}"
        final_rows.append(r)

    total_row = {"Site": "Total", "Total, n(%)": f"{overall_total} (100%)"}
    for cat_label, cat_vals in categories.items():
        n_total = df["Standard POCT Results"].isin(cat_vals).sum()
        pct_total = 100 * n_total / overall_total
        total_row[f"{cat_label} n(%)"] = f"{n_total} ({pct_total:.1f}%)"
    total_row["p-value"] = ""
    final_rows.append(total_row)

    col_order = [
        "Site",
        "Total, n(%)",
        "SCD n(%)",
        "SCT n(%)",
        "Normal n(%)",
        "Other Hb types n(%)",
        "p-value"
    ]

    return pd.DataFrame(final_rows)[col_order]

In [ ]:
df_all_prelim.head()

In [ ]:
df_all_prelim.columns

In [134]:
df_all_prelim.shape

(705, 10)

In [135]:
[c for c in df_all_prelim.columns if 'result' in c.lower()]

['Standard POCT Results', 'DBS POCT Results', 'Results']

In [136]:
df_all_prelim[['Results']].head(3)

,Results
0,FA - NO abnormal hemoglobin
1,FAC - C trait (AC)
2,FA - NO abnormal hemoglobin


In [137]:
hemoglobin_pattern_map = {
    1:'FA - NO abnormal hemoglobin',
    2:'AF - NO abnormal hemoglobin',
    3:'FS - Sickle cell disease SS (SCD-SS)/Sickle cell disease beta-0-thalassemia',
    4:'FSC - Sickle cell disease SC (SCD-SC)',
    5:'FSA - Sickle cell disease beta (+) thalassemia [SCD-beta(+)thal]',
    6:'FAS - Sickle cell trait (AS)',
    7:'FAC - C trait (AC)',
    8:'Other'
}

for c in ['Results']:
    recode_numeric(df_all_prelim, c, hemoglobin_pattern_map)
    
hemoglobin_to_genotype = {
    'FA - NO abnormal hemoglobin': 'AA',
    'FA': 'AA',
    'Fa': 'AA',
    'AF - NO abnormal hemoglobin': 'AA',
    'FAS - Sickle cell trait (AS)': 'AS',
    'FAS': 'AS',
    'FAC - C trait (AC)': 'AC',
    'FSC - Sickle cell disease SC (SCD-SC)': 'SC',
    'FS': 'SS/Sβ0',
    'FS - Sickle cell disease SS (SCD-SS)/Sickle cell disease beta-0-thalassemia': 'SS/Sβ0',
    'FSA - Sickle cell disease beta (+) thalassemia [SCD-beta(+)thal]': 'Sβ+',
    'Other': 'Other'
}

for c in ['Results']:
    df_all_prelim[c] = df_all_prelim[c].replace(hemoglobin_to_genotype)

In [138]:
df_all_prelim.head()

,Study ID,Facility ID,Study participant ID,Age,Sex,Standard POCT Results,DBS POCT Results,Results,confirmatory_test,country
0,0001,Kumasi,GH0001,1,Male,AA,AA,AA,ief,Ghana
1,0002,Kumasi,GH0002,1,Female,AC,AA,AC,ief,Ghana
2,0003,Kumasi,GH0003,1,Male,AA,AA,AA,ief,Ghana
3,0004,Kumasi,GH0004,42,Female,AS,AS,AS,ief,Ghana
4,0101,Kumasi,GH0101,43,Female,AA,AS,AS,ief,Ghana


In [139]:
df_all_prelim['Standard POCT Results'].value_counts(dropna=False)

Standard POCT Results
AA               337
1                247
AS                51
2                 41
AC                 6
6                  5
SS                 4
3                  4
7                  4
Indeterminate      3
4                  3
Name: count, dtype: int64

In [140]:
def recode_numeric(df, col, mapping):
    df[col] = df[col].replace({**mapping, **{float(k): v for k, v in mapping.items()}})
    return df

std_map = {1:'AA',2:'AS',3:'AC',4:'SS',5:'SC',6:'CC',7:'Indeterminate'}

for c in ['Standard POCT Results','DBS POCT Results']:
    recode_numeric(df_all_prelim, c, std_map)
    
df_all_prelim['Standard POCT Results'].value_counts(dropna=False)

Standard POCT Results
AA               584
AS                92
AC                10
Indeterminate      7
SS                 7
CC                 5
Name: count, dtype: int64

In [141]:
df_all = df_all_prelim[["country", "Standard POCT Results"]]
df_all.head()

,country,Standard POCT Results
0,Ghana,AA
1,Ghana,AC
2,Ghana,AA
3,Ghana,AS
4,Ghana,AA


In [142]:
df_all.shape

(705, 2)

In [143]:
table5 = make_poct_prevalence_table(df_all)
table5

,Site,"Total, n(%)",SCD n(%),SCT n(%),Normal n(%),Other Hb types n(%),p-value
0,Ghana,101 (14.3%),1 (1.0%),17 (16.8%),75 (74.3%),6 (5.9%),0.0000
1,Mali,104 (14.8%),0 (0.0%),5 (4.8%),87 (83.7%),8 (7.7%),0.0000
2,Nigeria,100 (14.2%),2 (2.0%),22 (22.0%),76 (76.0%),0 (0.0%),0.0000
3,Tanzania,100 (14.2%),1 (1.0%),14 (14.0%),84 (84.0%),1 (1.0%),0.0000
4,Uganda,100 (14.2%),3 (3.0%),20 (20.0%),76 (76.0%),0 (0.0%),0.0000
5,Zambia,100 (14.2%),0 (0.0%),9 (9.0%),91 (91.0%),0 (0.0%),0.0000
6,Zimbabwe,100 (14.2%),0 (0.0%),5 (5.0%),95 (95.0%),0 (0.0%),0.0000
7,Total,705 (100%),7 (1.0%),92 (13.0%),584 (82.8%),15 (2.1%),


In [147]:
from docx import Document
from docx.shared import Inches

doc = Document()
doc.add_heading('Table 5', level=1)

# Add table
table = doc.add_table(rows=1, cols=len(table5.columns))
table.style = 'Light Shading Accent 1'

hdr_cells = table.rows[0].cells
for i, col_name in enumerate(table5.columns):
    hdr_cells[i].text = str(col_name)

for _, row in table5.iterrows():
    row_cells = table.add_row().cells
    for i, value in enumerate(row):
        row_cells[i].text = str(value)

doc.save("Prevalence_tables/table_5_prelim.docx")

In [148]:
df_all_quant.head()

,record_id,facility_id,participant_id,country,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
1,2,GH001,GH2,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3.0,2024-02-19,Female,Strongly Agree,Disagree,Agree,Strongly Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,AC,AA,AC,NaN,NaN,1
2,3,GH001,GH3,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,32,1991-09-28,Female,Frafra,Mother,NaN,Married,Primary,NaN,Others,Seamstress,Islam,2.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
3,4,GH001,GH4,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,26,NaT,Female,Fulani,Mother,NaN,Married,Tertiary,NaN,Others,Seamstress,Islam,1.0,2024-01-09,Female,Strongly Agree,Agree,Strongly Agree,Disagree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,No,Yes,Yes,NaN,AS,AS,AS,NaN,NaN,42
4,5,GH001,GH5,Ghana,At Birthing Facility,2024-02-21,2024-02-21,21 02 2024,31,1993-01-01,Female,Songhai,Mother,NaN,Married,No education,NaN,Trader,NaN,Islam,7.0,2024-02-21,Male,Agree,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Agree,Agree,Agree,Agree,Agree,Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,0


In [149]:
df_all_quant.shape

(7496, 43)

In [150]:
df_all_quant['std_poct_results'].value_counts(dropna=False)

std_poct_results
NaN              6394
AA                912
AS                114
AC                 37
CC                 18
SS                  9
Indeterminate       8
SC                  4
Name: count, dtype: int64

In [151]:
df_all_quant[df_all_quant['country'] == 'Ghana']['std_poct_results'].value_counts(dropna=False)

std_poct_results
NaN              898
AA                75
AS                17
AC                 6
Indeterminate      2
SS                 1
Name: count, dtype: int64

In [152]:
df_all_quant[df_all_quant['country'] == 'Nigeria']['std_poct_results'].value_counts(dropna=False)

std_poct_results
NaN    2407
Name: count, dtype: int64

In [153]:
df_all_quant[df_all_quant['country'] == 'Mali']['std_poct_results'].value_counts(dropna=False)

std_poct_results
NaN              476
AA               433
AS                43
AC                30
CC                17
Indeterminate      5
SC                 4
SS                 3
Name: count, dtype: int64

In [154]:
df_all_quant[df_all_quant['country'] == 'Uganda']['std_poct_results'].value_counts(dropna=False)

std_poct_results
NaN              868
AA                85
AS                20
SS                 4
Indeterminate      1
CC                 1
Name: count, dtype: int64

In [155]:
df_all_quant[df_all_quant['country'] == 'Zimbabwe']['std_poct_results'].value_counts(dropna=False)

std_poct_results
NaN    450
AA      95
AS       5
Name: count, dtype: int64

In [156]:
df_all_quant[df_all_quant['country'] == 'Zambia']['std_poct_results'].value_counts(dropna=False)

std_poct_results
NaN    395
AA     140
AS      15
Name: count, dtype: int64

In [157]:
df_all_quant[df_all_quant['country'] == 'Tanzania']['std_poct_results'].value_counts(dropna=False)

std_poct_results
NaN    900
AA      84
AS      14
SS       1
AC       1
Name: count, dtype: int64

In [158]:
df_all_quant.head()

,record_id,facility_id,participant_id,country,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
1,2,GH001,GH2,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3.0,2024-02-19,Female,Strongly Agree,Disagree,Agree,Strongly Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,AC,AA,AC,NaN,NaN,1
2,3,GH001,GH3,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,32,1991-09-28,Female,Frafra,Mother,NaN,Married,Primary,NaN,Others,Seamstress,Islam,2.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
3,4,GH001,GH4,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,26,NaT,Female,Fulani,Mother,NaN,Married,Tertiary,NaN,Others,Seamstress,Islam,1.0,2024-01-09,Female,Strongly Agree,Agree,Strongly Agree,Disagree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,No,Yes,Yes,NaN,AS,AS,AS,NaN,NaN,42
4,5,GH001,GH5,Ghana,At Birthing Facility,2024-02-21,2024-02-21,21 02 2024,31,1993-01-01,Female,Songhai,Mother,NaN,Married,No education,NaN,Trader,NaN,Islam,7.0,2024-02-21,Male,Agree,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Agree,Agree,Agree,Agree,Agree,Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,0


In [159]:
df_all_quant['nbs_screening_locale'].value_counts(dropna=False)

nbs_screening_locale
NaN                         3106
At Immunisation Facility    2543
At Birthing Facility        1847
Name: count, dtype: int64

In [160]:
df_all_quant[df_all_quant['country'] == 'Mali']['religion'].value_counts(dropna=False)

religion
Islam           993
Christianity     16
<NA>              2
Name: count, dtype: int64

In [161]:
df_gh_quant = pd.read_csv('../data/raw/ghana/raw/ghana_phase_2.csv')
df_ml_quant = pd.read_excel('../data/raw/mali/raw/mali_phase_2.xlsx', sheet_name='NewBornScreening_DATA__20')
df_tn_quant = pd.concat([pd.read_csv('../data/raw/tanzania/raw/tanzania_prelim.csv'), pd.read_csv('../data/raw/tanzania/raw/tanzania_phase_2.csv')], ignore_index=True)
df_ug_quant = pd.read_csv('../data/raw/uganda/raw/uganda_phase_2.csv')
df_zim_quant = pd.concat([pd.read_csv('../data/raw/zim_zam/raw/zim.csv'), pd.read_csv('../data/raw/zim_zam/raw/zim_phase_2.csv')], ignore_index=True)
df_zam_quant = pd.read_csv('../data/raw/zim_zam/raw/zam_phase_2.csv')
df_ng_quant = pd.read_csv('../data/raw/nigeria/raw/nigeria_phase_2.csv') 
df_ng_prelim = pd.read_csv('../data/raw/nigeria/raw/prelims.csv') 
keep = [c for c in df_ng_quant.columns if c in df_ng_prelim.columns] 
df_ng_quant = pd.concat([df_ng_prelim[keep], df_ng_quant], ignore_index=True)

print('df_gh_quant:', df_gh_quant.shape)
print('df_ml_quant:', df_ml_quant.shape)
print('df_tn_quant:', df_tn_quant.shape)
print('df_ug_quant:', df_ug_quant.shape)
print('df_zim_quant:', df_zim_quant.shape)
print('df_zam_quant:', df_zam_quant.shape)
print('df_ng_quant:', df_ng_quant.shape)

df_gh_quant: (999, 89)
df_ml_quant: (1011, 92)
df_tn_quant: (1000, 52)
df_ug_quant: (980, 88)
df_zim_quant: (550, 51)
df_zam_quant: (550, 47)
df_ng_quant: (2507, 82)


In [162]:
df_all_quant.head()

,record_id,facility_id,participant_id,country,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
1,2,GH001,GH2,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3.0,2024-02-19,Female,Strongly Agree,Disagree,Agree,Strongly Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,AC,AA,AC,NaN,NaN,1
2,3,GH001,GH3,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,32,1991-09-28,Female,Frafra,Mother,NaN,Married,Primary,NaN,Others,Seamstress,Islam,2.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
3,4,GH001,GH4,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,26,NaT,Female,Fulani,Mother,NaN,Married,Tertiary,NaN,Others,Seamstress,Islam,1.0,2024-01-09,Female,Strongly Agree,Agree,Strongly Agree,Disagree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,No,Yes,Yes,NaN,AS,AS,AS,NaN,NaN,42
4,5,GH001,GH5,Ghana,At Birthing Facility,2024-02-21,2024-02-21,21 02 2024,31,1993-01-01,Female,Songhai,Mother,NaN,Married,No education,NaN,Trader,NaN,Islam,7.0,2024-02-21,Male,Agree,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Agree,Agree,Agree,Agree,Agree,Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,0


In [163]:
df_all_quant.shape

(7496, 43)

In [164]:
df_all_quant['country'].value_counts(dropna=False)

country
Nigeria     2407
Mali        1011
Tanzania    1000
Ghana        999
Uganda       979
Zimbabwe     550
Zambia       550
Name: count, dtype: int64

In [168]:
df_gh_quant.head(2)

,record_id,facility_id_screening,nbs_screening_locale,participant_id,date_enrolled,date_of_visit,dob_guardian,pgage_yrs,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,comments,screening_date,kind_of_screening_test___1,kind_of_screening_test___2,kind_of_screening_test___3,kind_of_screening_test___4,kind_of_screening_test___5,kind_of_screening_test___6,alt_screening_test,tests___1,tests___2,tests___3,tests___4,tests___5,std_poct_done,std_poct_test_date,std_poct_test_result_date,std_poct_results,dbspoct_test_date,dbspoct_test_result_date,dbspoct_test_results,ief_test_date,ief_test_result_date,ief_test_results,hplc_test_date,hplc_result_date,hplc_results,additional_molecular_test,additional_test_date,add_molecular_result_date,add_molecula_test_results,newborn_registryid,reminder_1,reminder1_method___1,reminder1_method___2,reminder1_method___3,reminder1_method___4,alt_reminder1_method,reminder1_personnel,reminder_2,reminder2_method___1,reminder2_method___2,reminder2_method___3,reminder2_method___4,alt_reminder2_method,reminder2_personnel,reminder_3,reminder3_method___1,reminder3_method___2,reminder3_method___3,reminder3_method___4,alt_reminder3_method,reminder3_personnel,subject_comments
0,1,GH001,1,ML001,20 02 2024,20 02 2024,12 10 1996,NaN,1,Akan,2.0,NaN,4.0,3.0,NaN,5.0,Decorator,Christian,1.0,19 02 2024,0.0,3.0,2.0,4.0,5.0,5.0,4.0,5.0,5.0,4.0,5.0,2.0,1.0,1.0,NaN,No,20 02 2024,1,0,0,0,0,0,NaN,1,1,1,0,0,1.0,20 02 2024,20 02 2024,1.0,20 02 2024,28 03 2024,1.0,19 03 2024,19 03 2024,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,NaN,NaN,NaN,0,0,0,0,NaN,NaN,NaN,0,0,0,0,NaN,NaN,NaN
1,2,GH001,1,ML002,20 02 2024,20 02 2024,NaN,NaN,1,Saliga,2.0,NaN,3.0,3.0,NaN,5.0,Seamstress,Muslim,3.0,19 02 2024,1.0,5.0,2.0,4.0,5.0,5.0,3.0,5.0,5.0,5.0,5.0,3.0,1.0,1.0,NaN,No,20 02 2024,1,0,0,0,0,0,NaN,1,1,1,0,0,1.0,20 02 2024,20 02 2024,3.0,20 02 2024,28 03 2024,1.0,19 03 2024,19 03 2024,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,NaN,NaN,NaN,0,0,0,0,NaN,NaN,NaN,0,0,0,0,NaN,NaN,NaN


In [169]:
def make_facility_table(df):

    country_order = ["Ghana", "Mali", "Nigeria", "Tanzania",
                     "Uganda", "Zambia", "Zimbabwe"]

    total_n = len(df)
    rows = []

    for country in country_order:
        if country not in df["country"].unique():
            continue

        rows.append({
            "Site facility": country,
            "Frequency, n(%)": ""
        })

        sub = df[df["country"] == country]

        counts = sub["facility_id"].value_counts(dropna=False)

        for facility, n in counts.items():
            pct = n / total_n * 100
            rows.append({
                "Site facility": f"   {facility}",
                "Frequency, n(%)": f"{n} ({pct:.1f}%)"
            })

    return pd.DataFrame(rows)

In [170]:
facility_table = make_facility_table(df_all_quant)

In [171]:
df_all_quant.head(2)

,record_id,facility_id,participant_id,country,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
1,2,GH001,GH2,Ghana,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3.0,2024-02-19,Female,Strongly Agree,Disagree,Agree,Strongly Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,AC,AA,AC,NaN,NaN,1


In [172]:
df_all_quant['country'].value_counts(dropna=False)

country
Nigeria     2407
Mali        1011
Tanzania    1000
Ghana        999
Uganda       979
Zimbabwe     550
Zambia       550
Name: count, dtype: int64

In [173]:
df_all_quant['facility_name'] = ''
df_all_quant.insert(
    4,
    'facility_name',
    df_all_quant.pop('facility_name')
)

In [174]:
df_all_quant.head(2)

,record_id,facility_id,participant_id,country,facility_name,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
1,2,GH001,GH2,Ghana,,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3.0,2024-02-19,Female,Strongly Agree,Disagree,Agree,Strongly Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,AC,AA,AC,NaN,NaN,1


In [175]:
df_all_quant.loc[df_all_quant['country'].eq('Mali'), 'facility_id'] = 'ML001'

df_all_quant.loc[
    df_all_quant['country'].eq('Mali'),
    'facility_name'
] = 'CSCOM Yirimadio (Bamako)'

In [176]:
df_all_quant[df_all_quant['country'] == 'Mali']['facility_id'].value_counts(dropna=False)

facility_id
ML001    1011
Name: count, dtype: int64

In [177]:
df_all_quant.head(2)

,record_id,facility_id,participant_id,country,facility_name,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
1,2,GH001,GH2,Ghana,,At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3.0,2024-02-19,Female,Strongly Agree,Disagree,Agree,Strongly Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,AC,AA,AC,NaN,NaN,1


In [178]:
df_all_quant[df_all_quant['country'] == 'Ghana']['facility_id'].value_counts(dropna=False)

facility_id
GH001     992
GH002       2
 GH001      1
MR313       1
G           1
NaN         1
MP115       1
Name: count, dtype: int64

In [179]:
df_all_quant.loc[df_all_quant['country'].eq('Ghana'), 'facility_id'] = 'GH001'

df_all_quant.loc[
    df_all_quant['country'].eq('Ghana'),
    'facility_name'
] = 'Manhyia District Hospital (MDH)'

In [180]:
df_all_quant[df_all_quant['country'] == 'Ghana']['facility_id'].value_counts(dropna=False)

facility_id
GH001    999
Name: count, dtype: int64

In [181]:
df_all_quant.head(2)

,record_id,facility_id,participant_id,country,facility_name,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,Manhyia District Hospital (MDH),At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
1,2,GH001,GH2,Ghana,Manhyia District Hospital (MDH),At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3.0,2024-02-19,Female,Strongly Agree,Disagree,Agree,Strongly Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,AC,AA,AC,NaN,NaN,1


In [182]:
df_all_quant[df_all_quant['country'] == 'Uganda']['facility_id'].value_counts(dropna=False)

facility_id
LRRH             957
Lrrh              10
LRRH               4
LRR                1
LRRA               1
LR4H               1
 LRRH              1
LRTH               1
LLRH               1
LRRH.              1
LRRH/NBS/0633      1
Name: count, dtype: int64

In [183]:
df_all_quant.loc[df_all_quant['country'].eq('Uganda'), 'facility_id'] = 'UG001'
df_all_quant.loc[df_all_quant['country'].eq('Uganda'), 'facility_name'] = 'Lira Regional Referral Hospital (LRRH)'

df_all_quant[df_all_quant['country'].eq('Uganda')]['facility_id'].value_counts(dropna=False)

facility_id
UG001    979
Name: count, dtype: int64

In [184]:
df_all_quant[df_all_quant['country'] == 'Uganda']['facility_id'].value_counts(dropna=False)

facility_id
UG001    979
Name: count, dtype: int64

In [185]:
df_all_quant[df_all_quant['country'] == 'Uganda']['facility_name'].value_counts(dropna=False)

facility_name
Lira Regional Referral Hospital (LRRH)    979
Name: count, dtype: int64

In [186]:
df_all_quant.head(2)

,record_id,facility_id,participant_id,country,facility_name,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,Manhyia District Hospital (MDH),At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
1,2,GH001,GH2,Ghana,Manhyia District Hospital (MDH),At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3.0,2024-02-19,Female,Strongly Agree,Disagree,Agree,Strongly Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,AC,AA,AC,NaN,NaN,1


In [187]:
df_all_quant[df_all_quant['country'] == 'Tanzania']['facility_id'].value_counts(dropna=False)

facility_id
BMC-TZ     535
BMC-TZ     465
Name: count, dtype: int64

In [188]:
df_all_quant['facility_id'] = (
    df_all_quant['facility_id']
    .astype(str)
    .str.strip()
)

In [189]:
df_all_quant[df_all_quant['country']=='Tanzania']['facility_id'].value_counts(dropna=False)

facility_id
BMC-TZ    1000
Name: count, dtype: int64

In [190]:
df_all_quant.loc[df_all_quant['country'].eq('Tanzania'), 'facility_id'] = 'TZ001'

df_all_quant.loc[df_all_quant['country'].eq('Tanzania'), 'facility_name'] = 'Bugando Medical Centre (BMC), Tanzania'

df_all_quant[df_all_quant['country'].eq('Tanzania')]['facility_id'].value_counts(dropna=False)

facility_id
TZ001    1000
Name: count, dtype: int64

In [191]:
df_all_quant[df_all_quant['country'] == 'Tanzania']['facility_name'].value_counts(dropna=False)

facility_name
Bugando Medical Centre (BMC), Tanzania    1000
Name: count, dtype: int64

In [192]:
df_all_quant.head(2)

,record_id,facility_id,participant_id,country,facility_name,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,Manhyia District Hospital (MDH),At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
1,2,GH001,GH2,Ghana,Manhyia District Hospital (MDH),At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3.0,2024-02-19,Female,Strongly Agree,Disagree,Agree,Strongly Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,AC,AA,AC,NaN,NaN,1


In [193]:
df_all_quant[df_all_quant['country'] == 'Zambia']['facility_id'].value_counts(dropna=False)

facility_id
UTH     254
MFLH    195
nan     100
Uth       1
Name: count, dtype: int64

In [194]:
df_all_quant.loc[(df_all_quant['country'] == 'Zambia') &(df_all_quant['facility_id'] == 'UTH'),'facility_id'] = 'ZM001'
df_all_quant.loc[(df_all_quant['country'] == 'Zambia') &(df_all_quant['facility_id'] == 'Uth'),'facility_id'] = 'ZM001'
df_all_quant.loc[(df_all_quant['country'] == 'Zambia') &(df_all_quant['facility_id'] == 'MFLH'),'facility_id'] = 'ZM002'

In [195]:
df_all_quant.loc[df_all_quant['facility_id'] == 'ZM001','facility_name'] = 'University Teaching Hospital (UTH), Lusaka'

df_all_quant.loc[df_all_quant['facility_id'] == 'ZM002','facility_name'] = 'Levy Mwanawasa General Hospital (LMGH), Lusaka'

In [196]:
df_all_quant[df_all_quant['country'] == 'Zambia']['facility_id'].value_counts(dropna=False)

facility_id
ZM001    255
ZM002    195
nan      100
Name: count, dtype: int64

In [197]:
df_all_quant[df_all_quant['country'] == 'Zambia']['facility_name'].value_counts(dropna=False)

facility_name
University Teaching Hospital (UTH), Lusaka        255
Levy Mwanawasa General Hospital (LMGH), Lusaka    195
                                                  100
Name: count, dtype: int64

In [198]:
df_all_quant.head(2)

,record_id,facility_id,participant_id,country,facility_name,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,Manhyia District Hospital (MDH),At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
1,2,GH001,GH2,Ghana,Manhyia District Hospital (MDH),At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3.0,2024-02-19,Female,Strongly Agree,Disagree,Agree,Strongly Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,AC,AA,AC,NaN,NaN,1


In [199]:
df_all_quant[df_all_quant['country'] == 'Zimbabwe']['facility_id'].value_counts(dropna=False)

facility_id
PTH    302
MTH    248
Name: count, dtype: int64

In [200]:
df_all_quant.loc[(df_all_quant['country'] == 'Zimbabwe') &(df_all_quant['facility_id'] == 'PTH'), 'facility_id'] = 'ZW001'

df_all_quant.loc[(df_all_quant['country'] == 'Zimbabwe') &(df_all_quant['facility_id'] == 'MTH'),'facility_id'] = 'ZW002'

df_all_quant.loc[df_all_quant['facility_id'] == 'ZW001','facility_name'] = 'Parirenyatwa Group of Hospitals (PTH), Harare'

df_all_quant.loc[df_all_quant['facility_id'] == 'ZW002','facility_name'] = 'Mutoko District Hospital, Zimbabwe'

In [201]:
df_all_quant[df_all_quant['country'] == 'Zimbabwe']['facility_id'].value_counts(dropna=False)

facility_id
ZW001    302
ZW002    248
Name: count, dtype: int64

In [202]:
df_all_quant[df_all_quant['country'] == 'Zimbabwe']['facility_name'].value_counts(dropna=False)

facility_name
Parirenyatwa Group of Hospitals (PTH), Harare    302
Mutoko District Hospital, Zimbabwe               248
Name: count, dtype: int64

In [203]:
df_all_quant.head(2)

,record_id,facility_id,participant_id,country,facility_name,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,Manhyia District Hospital (MDH),At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
1,2,GH001,GH2,Ghana,Manhyia District Hospital (MDH),At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3.0,2024-02-19,Female,Strongly Agree,Disagree,Agree,Strongly Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,AC,AA,AC,NaN,NaN,1


In [204]:
df_all_quant[df_all_quant['country'] == 'Nigeria']['facility_id'].value_counts(dropna=False)

facility_id
PHC AZUIYIOKWU                    100
NAUTH                             100
U M T H                           100
WGH                               100
NGH                               100
ALIMOSHO                          100
RSUTH                              99
FMC Keffi                          99
Ahmadu Bello Teaching Hospital     96
AKTH                               96
FMC BK 001                         96
ZANKLI MEDICAL SERVICES            96
OOUTH                              94
GME                                90
UNTH/UWANI HEALTH CENTRE           89
UUTH/WIPHC                         82
ASABA/UMUAGU                       78
PARKLANE ENUGU                     76
BDTH/URM                           74
ISTH/PHC Ujoelen                   56
IDI-OGUNGUN                        54
ISTH                               44
NBS/NIG/OAUTHC/IFELODUN            39
NHA                                37
GIRI                               33
UATH                               29


In [205]:
import re
import numpy as np

ng_recode = {
    # FMC KEFFI variants
    "FMC KEFFI": "FMC_KEFFI",
    "FMC KEFFI ": "FMC_KEFFI",
    "FMC KEFFI".title(): "FMC_KEFFI",
    "FMC KEFFI".capitalize(): "FMC_KEFFI",
    "FMC KEFFI".lower(): "FMC_KEFFI",

    "AHMADU BELLO TEACHING HOSPITAL": "ABUTH",
    "Ahmadu Bello Teaching Hospital".upper(): "ABUTH",
    "Ahmadu Bello Teaching Hospital": "ABUTH",

    "U M T H": "UMTH",
    "UMTH": "UMTH",

    # FMC BK variants
    "FMC BK 001": "FMC_BK",
    "FMC BK 003": "FMC_BK",
    "FMC BK": "FMC_BK",

    # Zankli variants
    "ZANKLI MEDICAL SERVICES": "ZANKLI",
    "ZANKLI MEDICAL CENTRE": "ZANKLI",

    # Parklane variants
    "PARKLANE ENUGU": "PARKLANE_ENUGU",
    "PARKLANE Enugu": "PARKLANE_ENUGU",

    # GIRI variants + NBS prefixed GIRI
    "GIRI": "GIRI_PHC",
    "Giri": "GIRI_PHC",
    "Giri PHC": "GIRI_PHC",
    "Giri 036": "GIRI_PHC",
    "NBS/FCT/GIRI": "GIRI_PHC",
    "NBS/FCT/ GIRI": "GIRI_PHC",
    "NBS/ FCT/GIRI": "GIRI_PHC",

    # GWAKO variants
    "Gwako": "GWAKO",
    "GWAKO": "GWAKO",

    # OAUTHC variants (keep sub-site as part of ID)
    "NBS/NIG/OAUTHC/IFELODUN": "OAUTHC_IFELODUN",
    "NBS/NIG/OAUTH/IFELODUN": "OAUTHC_IFELODUN",
    "JNBS/NIG/OAUTHC/IFELODUN": "OAUTHC_IFELODUN",
    "NBS/NIG/OAUTHC/ENUWA": "OAUTHC_ENUWA",
    "NBS/NIG/OAUTH/ENUWA": "OAUTHC_ENUWA",
    "NBS/NIG/OAUTHC/MOORE": "OAUTHC_MOORE",
    "NBS/NIG/OAUTH/MOORE": "OAUTHC_MOORE",

    # UNTH/Uwani variants
    "UNTH/UWANI HEALTH CENTRE": "UNTH_UWANI_HC",
    "UNTH/UWANI HEALTH  CENTRE": "UNTH_UWANI_HC",
    "UNTH/ UWANI HEALTH  CENTRE": "UNTH_UWANI_HC",
    "UNTH/ Uwani Health  Centre": "UNTH_UWANI_HC",
    "UNTH/Uwani Health Centre": "UNTH_UWANI_HC",
    "UNTH/UWANI HEALTH8 CENTRE": "UNTH_UWANI_HC",

    # JUTH variants (collapse all the numbered ones)
    "JUTH-PHC-TOWNSHIP": "JUTH_TOWNSHIP",
    "JUTH-PHC TOWNSHIP": "JUTH_TOWNSHIP",
    "JUTH- PHC TOWNSHIP": "JUTH_TOWNSHIP",
    "JUTH -PHC TOWNSHIP": "JUTH_TOWNSHIP",
    "JUTH-TOWNSHIP": "JUTH_TOWNSHIP",
    "JUTH-PHC TOWNSHIP 095": "JUTH_TOWNSHIP",
}

In [206]:
def ng_canonicalize_facility_id(x):
    if pd.isna(x): 
        return np.nan
    s = str(x).strip()

    s2 = s.strip()

    if re.search(r"JUTH", s2, re.I) and re.search(r"TOWNSHIP", s2, re.I):
        return "JUTH_TOWNSHIP"

    if re.search(r"JUTH", s2, re.I) and re.search(r"ANGWANRIMI", s2, re.I):
        return "JUTH_ANGWANRIMI"

    if re.search(r"JUTH", s2, re.I) and re.search(r"ANGWANROGO", s2, re.I):
        return "JUTH_ANGWANROGO"

    s2 = re.sub(r"[- ]\d{2,4}$", "", s2)

    return ng_recode.get(s2, ng_recode.get(s2.upper(), s2.upper()))

mask_ng = df_all_quant['country'].eq('Nigeria')
df_all_quant.loc[mask_ng, 'facility_id'] = df_all_quant.loc[mask_ng, 'facility_id'].apply(ng_canonicalize_facility_id)

In [207]:
ng_facility_lookup = {
    "PHC_AZUIYIOKWU": "PHC Azuikwokwu (Azuikwokwu PHC)",
    "UMTH": "University of Maiduguri Teaching Hospital (UMTH)",
    "WGH": "WGH (confirm full name)",
    "NGH": "NGH (confirm full name)",
    "NAUTH": "Nnamdi Azikiwe University Teaching Hospital (NAUTH)",
    "ALIMOSHO": "Alimosho (PHC/Clinic)",
    "RSUTH": "Rivers State University Teaching Hospital (RSUTH)",
    "FMC_KEFFI": "Federal Medical Centre, Keffi",
    "ABUTH": "Ahmadu Bello University Teaching Hospital (ABUTH)",
    "AKTH": "Aminu Kano Teaching Hospital (AKTH)",
    "FMC_BK": "Federal Medical Centre, Birnin Kebbi",
    "ZANKLI": "Zankli Medical Services",
    "OOUTH": "Olabisi Onabanjo University Teaching Hospital (OOUTH)",
    "GME": "GME (confirm full name)",
    "UNTH_UWANI_HC": "UNTH / Uwani Health Centre (Enugu)",
    "UUTH_WIPHC": "UUTH / WIPHC (confirm full name)",
    "ASABA_UMUAGU": "Asaba / Umuagu (Clinic/Hospital)",
    "PARKLANE_ENUGU": "Parklane Hospital, Enugu",
    "BDTH_URM": "BDTH / URM (confirm full name)",
    "ISTH_PHC_UJOELEN": "ISTH / PHC Ujoelen (confirm full name)",
    "IDI_OGUNGUN": "Idi-Ogungun (PHC/Clinic)",
    "ISTH": "ISTH (confirm full name)",
    "OAUTHC_IFELODUN": "OAUTHC (Ifelodun)",
    "OAUTHC_ENUWA": "OAUTHC (Enuwa)",
    "OAUTHC_MOORE": "OAUTHC (Moore)",
    "NHA": "NHA (confirm full name)",
    "GIRI_PHC": "Giri PHC (FCT)",
    "UATH": "UATH (confirm full name)",
    "PAIKO": "Paiko (Niger State/FCT area)",
    "GWAKO": "Gwako (FCT)",
    "JUTH_TOWNSHIP": "JUTH PHC Township",
    "BDTH_BDW": "BDTH / BDW (confirm full name)",
    "ASABA_ST_JOSEPH": "Asaba / St. Joseph (Clinic/Hospital)",
    "UUTH": "UUTH (confirm full name)",
    "AGBOWO": "Agbowo (PHC/Clinic)",
    "GWA": "Gwa (FCT)",
    "JUTH_ANGWANRIMI": "JUTH PHC Angwanrimi",
    "BASHORUN": "Bashorun (PHC/Clinic)",
    "SANGO": "Sango (PHC/Clinic)",
    "GTC": "GTC (Clinic/PHC)",
    "JUTH_ANGWANROGO": "JUTH PHC Angwanrogo",
    "DAGIRI": "Dagiri (FCT)",
    "UNTH_ENUGU": "University of Nigeria Teaching Hospital (UNTH), Enugu",
}

In [208]:
facility_lookup = {
    # Ghana
    "GH001": "Manhyia District Hospital (MDH), Ghana",

    # Mali
    "ML001": "CSCOM Yirimadio (Bamako), Mali",

    # Uganda
    "UG001": "Lira Regional Referral Hospital (LRRH), Uganda",

    # Tanzania
    "TZ001": "Bugando Medical Centre (BMC), Tanzania",

    # Zambia
    "ZM001": "University Teaching Hospital (UTH), Lusaka",
    "ZM002": "Levy Mwanawasa General Hospital (LMGH), Lusaka",

    # Zimbabwe
    "ZW001": "Parirenyatwa Group of Hospitals (PTH), Harare",
    "ZW002": "Mutoko District Hospital, Zimbabwe",
}

In [209]:
facility_lookup.update(ng_facility_lookup)

In [210]:
df_all_quant['facility_name'] = df_all_quant['facility_id'].map(facility_lookup)

In [211]:
df_all_quant[df_all_quant['facility_name'].isna()][['country','facility_id']].drop_duplicates()

,country,facility_id
4540,Zambia,nan
5190,Nigeria,PHC AZUIYIOKWU
5386,Nigeria,BDTH/BDW
5408,Nigeria,BDTH/URM
5482,Nigeria,ASABA/UMUAGU
5486,Nigeria,ASB/ST JOSEPH
5489,Nigeria,ASABA/ST. JOSEPH
5490,Nigeria,ASABA/ST JOSEPH
5807,Nigeria,G
5860,Nigeria,JUTH-PHC ANGWA


In [212]:
df_all_quant.head()

,record_id,facility_id,participant_id,country,facility_name,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,"Manhyia District Hospital (MDH), Ghana",At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
1,2,GH001,GH2,Ghana,"Manhyia District Hospital (MDH), Ghana",At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3.0,2024-02-19,Female,Strongly Agree,Disagree,Agree,Strongly Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,AC,AA,AC,NaN,NaN,1
2,3,GH001,GH3,Ghana,"Manhyia District Hospital (MDH), Ghana",At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,32,1991-09-28,Female,Frafra,Mother,NaN,Married,Primary,NaN,Others,Seamstress,Islam,2.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
3,4,GH001,GH4,Ghana,"Manhyia District Hospital (MDH), Ghana",At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,26,NaT,Female,Fulani,Mother,NaN,Married,Tertiary,NaN,Others,Seamstress,Islam,1.0,2024-01-09,Female,Strongly Agree,Agree,Strongly Agree,Disagree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,No,Yes,Yes,NaN,AS,AS,AS,NaN,NaN,42
4,5,GH001,GH5,Ghana,"Manhyia District Hospital (MDH), Ghana",At Birthing Facility,2024-02-21,2024-02-21,21 02 2024,31,1993-01-01,Female,Songhai,Mother,NaN,Married,No education,NaN,Trader,NaN,Islam,7.0,2024-02-21,Male,Agree,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Agree,Agree,Agree,Agree,Agree,Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,0


In [213]:
df_all_quant[df_all_quant['country'] == 'Nigeria']['facility_id'].value_counts(dropna=False)

facility_id
PHC AZUIYIOKWU      100
FMC_BK              100
WGH                 100
ALIMOSHO            100
NAUTH               100
NGH                 100
UMTH                100
ZANKLI               99
RSUTH                99
FMC_KEFFI            99
AKTH                 96
ABUTH                96
UNTH_UWANI_HC        95
OOUTH                94
PARKLANE_ENUGU       92
GME                  90
UUTH/WIPHC           82
ASABA/UMUAGU         78
BDTH/URM             74
JUTH_TOWNSHIP        61
ISTH/PHC UJOELEN     56
IDI-OGUNGUN          54
ISTH                 44
OAUTHC_ENUWA         42
OAUTHC_IFELODUN      41
GIRI_PHC             37
NHA                  37
UATH                 29
GWAKO                26
JUTH_ANGWANRIMI      23
BDTH/BDW             22
ASABA/ST. JOSEPH     19
OAUTHC_MOORE         17
UUTH                 17
AGBOWO               16
BASHORUN             15
JUTH_ANGWANROGO      14
SANGO                14
GTC                  12
UNTH ENUGU            6
NBS/FCT/DAGIRI        6
ASB/

In [214]:
df_all_quant[df_all_quant['country'] == 'Nigeria'].head()

,record_id,facility_id,participant_id,country,facility_name,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
5190,AEFUTHA/001,PHC AZUIYIOKWU,NGAEFUTHA/001,Nigeria,NaN,At Immunisation Facility,NaT,NaT,19/02/2024,<NA>,NaT,Female,IGBO,Mother,NaN,Married,Secondary,NaN,Employed,NaN,Christianity,4.0,NaT,Male,Disagree,Disagree,Disagree,Strongly Disagree,Agree,Agree,Agree,Agree,Strongly Agree,Agree,Yes,Yes,Yes,NaN,NaN,AA,NaN,NaN,NaN,<NA>
5191,AEFUTHA/002,PHC AZUIYIOKWU,NGAEFUTHA/002,Nigeria,NaN,At Immunisation Facility,NaT,NaT,19/02/2024,<NA>,NaT,Female,IGBO,Mother,NaN,Married,Secondary,NaN,Housewife,NaN,Christianity,1.0,NaT,Female,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Yes,Yes,Yes,NaN,NaN,AA,NaN,NaN,NaN,<NA>
5192,AEFUTHA/003,PHC AZUIYIOKWU,NGAEFUTHA/003,Nigeria,NaN,At Immunisation Facility,NaT,NaT,19/02/2024,<NA>,NaT,Female,1GBO,Mother,NaN,Married,Secondary,NaN,Others,ARTISAN,Christianity,0.0,NaT,Female,Disagree,Disagree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Yes,Yes,Yes,NaN,NaN,AS,AS,NaN,NaN,<NA>
5193,AEFUTHA/004,PHC AZUIYIOKWU,NGAEFUTHA/004,Nigeria,NaN,At Immunisation Facility,NaT,NaT,19/02/2024,<NA>,NaT,Female,IGBO,Mother,NaN,Married,Secondary,NaN,Trader,NaN,Christianity,5.0,NaT,Female,Disagree,Disagree,Agree,Agree,Disagree,Agree,Agree,Agree,Agree,Agree,Yes,Yes,Yes,NaN,NaN,AS,AS,NaN,NaN,<NA>
5194,AEFUTHA/005,PHC AZUIYIOKWU,NGAEFUTHA/005,Nigeria,NaN,At Immunisation Facility,NaT,NaT,19/02/2024,<NA>,NaT,Female,IGBO,Mother,NaN,Married,Secondary,NaN,Others,ARTISAN,Christianity,0.0,NaT,Female,Disagree,Disagree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Yes,Yes,Yes,NaN,NaN,AA,NaN,NaN,NaN,<NA>


In [215]:
mask_ng = df_all_quant['country'].eq('Nigeria')

ng_lastmile = {
    "NBS/FCT/PAIKO": "PAIKO",
    "NBS/FCT/GWA": "GWA",
    "NBS/FCT/UATH": "UATH",
    "NBS/FCT/DAGIRI": "DAGIRI",

    "UNTH ENUGU": "UNTH_ENUGU",

    "ASABA/ST JOSEPH": "ASABA_ST_JOSEPH",
    "ASABA/ST. JOSEPH": "ASABA_ST_JOSEPH",
    "ASB/ST JOSEPH": "ASABA_ST_JOSEPH",
    
    "ASABA/UMUAGU": "ASABA_UMUAGU",

    "JUTH-PHC ANGWA": "JUTH_TOWNSHIP",

    "G": np.nan,
}

df_all_quant.loc[mask_ng, 'facility_id'] = df_all_quant.loc[mask_ng, 'facility_id'].replace(ng_lastmile)

In [216]:
mask_ng = df_all_quant['country'].eq('Nigeria')

df_all_quant.loc[
    mask_ng & df_all_quant['facility_id'].eq('UUTH/WIPHC'),
    'facility_id'
] = 'UUTH'

In [217]:
df_all_quant.loc[mask_ng & df_all_quant['facility_id'].eq('BDTH/URM'),
                 'facility_id'] = 'BDTH_URM'

df_all_quant.loc[mask_ng & df_all_quant['facility_id'].eq('BDTH/BDW'),
                 'facility_id'] = 'BDTH_BDW'

df_all_quant.loc[mask_ng & df_all_quant['facility_id'].eq('ISTH/PHC UJOELEN'),
                 'facility_id'] = 'ISTH_PHC_UJOELEN'

In [218]:
df_all_quant[df_all_quant['country'] == 'Nigeria']['facility_id'].value_counts(dropna=False)

facility_id
PHC AZUIYIOKWU      100
FMC_BK              100
WGH                 100
ALIMOSHO            100
UMTH                100
NGH                 100
NAUTH               100
UUTH                 99
RSUTH                99
ZANKLI               99
FMC_KEFFI            99
AKTH                 96
ABUTH                96
UNTH_UWANI_HC        95
OOUTH                94
PARKLANE_ENUGU       92
GME                  90
ASABA_UMUAGU         78
BDTH_URM             74
JUTH_TOWNSHIP        62
ISTH_PHC_UJOELEN     56
IDI-OGUNGUN          54
ISTH                 44
OAUTHC_ENUWA         42
OAUTHC_IFELODUN      41
GIRI_PHC             37
NHA                  37
UATH                 29
GWAKO                26
JUTH_ANGWANRIMI      23
BDTH_BDW             22
ASABA_ST_JOSEPH      22
OAUTHC_MOORE         17
AGBOWO               16
BASHORUN             15
SANGO                14
JUTH_ANGWANROGO      14
GTC                  12
DAGIRI                6
UNTH_ENUGU            6
NaN                   1
Name

In [219]:
df_all_quant[
    (df_all_quant['country']=='Nigeria') &
    (df_all_quant['facility_id'].isna())
]

,record_id,facility_id,participant_id,country,facility_name,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
5807,GME89,NaN,NGGME89,Nigeria,NaN,At Immunisation Facility,2024-07-02,NaT,07/02/2024,<NA>,NaT,Female,Fulani,Mother,NaN,Married,Secondary,NaN,Housewife,NaN,Islam,NaN,NaT,Male,Agree,Agree,Neutral/ I don't know,Agree,Agree,Agree,Agree,Agree,Agree,Agree,I don't know,Yes,Yes,NaN,NaN,AA,NaN,NaN,NaN,<NA>


In [220]:
facility_lookup.update(ng_facility_lookup)

In [221]:
df_all_quant['facility_name'] = df_all_quant['facility_id'].map(facility_lookup)

In [222]:
df_all_quant.head()

,record_id,facility_id,participant_id,country,facility_name,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,"Manhyia District Hospital (MDH), Ghana",At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
1,2,GH001,GH2,Ghana,"Manhyia District Hospital (MDH), Ghana",At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3.0,2024-02-19,Female,Strongly Agree,Disagree,Agree,Strongly Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,AC,AA,AC,NaN,NaN,1
2,3,GH001,GH3,Ghana,"Manhyia District Hospital (MDH), Ghana",At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,32,1991-09-28,Female,Frafra,Mother,NaN,Married,Primary,NaN,Others,Seamstress,Islam,2.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
3,4,GH001,GH4,Ghana,"Manhyia District Hospital (MDH), Ghana",At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,26,NaT,Female,Fulani,Mother,NaN,Married,Tertiary,NaN,Others,Seamstress,Islam,1.0,2024-01-09,Female,Strongly Agree,Agree,Strongly Agree,Disagree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,No,Yes,Yes,NaN,AS,AS,AS,NaN,NaN,42
4,5,GH001,GH5,Ghana,"Manhyia District Hospital (MDH), Ghana",At Birthing Facility,2024-02-21,2024-02-21,21 02 2024,31,1993-01-01,Female,Songhai,Mother,NaN,Married,No education,NaN,Trader,NaN,Islam,7.0,2024-02-21,Male,Agree,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Agree,Agree,Agree,Agree,Agree,Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,0


In [223]:
df_all_quant[df_all_quant['facility_name'].isna()][['country','facility_id']].drop_duplicates()

,country,facility_id
4540,Zambia,nan
5190,Nigeria,PHC AZUIYIOKWU
5807,Nigeria,NaN
7231,Nigeria,IDI-OGUNGUN


In [224]:
df_all_quant[df_all_quant['country'] == 'Nigeria'].head()

,record_id,facility_id,participant_id,country,facility_name,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
5190,AEFUTHA/001,PHC AZUIYIOKWU,NGAEFUTHA/001,Nigeria,NaN,At Immunisation Facility,NaT,NaT,19/02/2024,<NA>,NaT,Female,IGBO,Mother,NaN,Married,Secondary,NaN,Employed,NaN,Christianity,4.0,NaT,Male,Disagree,Disagree,Disagree,Strongly Disagree,Agree,Agree,Agree,Agree,Strongly Agree,Agree,Yes,Yes,Yes,NaN,NaN,AA,NaN,NaN,NaN,<NA>
5191,AEFUTHA/002,PHC AZUIYIOKWU,NGAEFUTHA/002,Nigeria,NaN,At Immunisation Facility,NaT,NaT,19/02/2024,<NA>,NaT,Female,IGBO,Mother,NaN,Married,Secondary,NaN,Housewife,NaN,Christianity,1.0,NaT,Female,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Yes,Yes,Yes,NaN,NaN,AA,NaN,NaN,NaN,<NA>
5192,AEFUTHA/003,PHC AZUIYIOKWU,NGAEFUTHA/003,Nigeria,NaN,At Immunisation Facility,NaT,NaT,19/02/2024,<NA>,NaT,Female,1GBO,Mother,NaN,Married,Secondary,NaN,Others,ARTISAN,Christianity,0.0,NaT,Female,Disagree,Disagree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Yes,Yes,Yes,NaN,NaN,AS,AS,NaN,NaN,<NA>
5193,AEFUTHA/004,PHC AZUIYIOKWU,NGAEFUTHA/004,Nigeria,NaN,At Immunisation Facility,NaT,NaT,19/02/2024,<NA>,NaT,Female,IGBO,Mother,NaN,Married,Secondary,NaN,Trader,NaN,Christianity,5.0,NaT,Female,Disagree,Disagree,Agree,Agree,Disagree,Agree,Agree,Agree,Agree,Agree,Yes,Yes,Yes,NaN,NaN,AS,AS,NaN,NaN,<NA>
5194,AEFUTHA/005,PHC AZUIYIOKWU,NGAEFUTHA/005,Nigeria,NaN,At Immunisation Facility,NaT,NaT,19/02/2024,<NA>,NaT,Female,IGBO,Mother,NaN,Married,Secondary,NaN,Others,ARTISAN,Christianity,0.0,NaT,Female,Disagree,Disagree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Agree,Yes,Yes,Yes,NaN,NaN,AA,NaN,NaN,NaN,<NA>


In [225]:
df_all_quant[df_all_quant['country'] == 'Nigeria'].tail()

,record_id,facility_id,participant_id,country,facility_name,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
7592,ZMC/095,ZANKLI,NGZMC/095,Nigeria,Zankli Medical Services,At Immunisation Facility,NaT,NaT,20/02/2024,<NA>,NaT,Female,NaN,Mother,NaN,Married,Secondary,NaN,Trader,NaN,Islam,NaN,NaT,Female,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,NaN,AA,NaN,NaN,NaN,<NA>
7593,ZMC/096,ZANKLI,NGZMC/096,Nigeria,Zankli Medical Services,At Immunisation Facility,NaT,NaT,20/02/2024,<NA>,NaT,Female,NaN,Mother,NaN,NaN,Secondary,NaN,NaN,NaN,Islam,NaN,NaT,Male,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,I don't know,Yes,Yes,NaN,NaN,AS,AS,NaN,NaN,<NA>
7594,ZMC/097,ZANKLI,NGZMC/097,Nigeria,Zankli Medical Services,At Immunisation Facility,NaT,NaT,20/02/2024,<NA>,NaT,Female,NaN,Mother,NaN,NaN,Primary,NaN,Trader,NaN,Christianity,NaN,NaT,Female,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,I don't know,Yes,Yes,NaN,NaN,AA,NaN,NaN,NaN,<NA>
7595,ZMC/098,ZANKLI,NGZMC/098,Nigeria,Zankli Medical Services,At Immunisation Facility,NaT,NaT,20/02/2024,<NA>,NaT,Female,NaN,Mother,NaN,Married,Secondary,NaN,Trader,NaN,Islam,NaN,NaT,Female,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,I don't know,Yes,Yes,NaN,NaN,AA,NaN,NaN,NaN,<NA>
7596,ZMC/099,ZANKLI,NGZMC/099,Nigeria,Zankli Medical Services,At Immunisation Facility,NaT,NaT,20/02/2024,<NA>,NaT,Female,NaN,Mother,NaN,NaN,Secondary,NaN,NaN,NaN,Christianity,NaN,NaT,Female,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,Neutral/ I don't know,I don't know,Yes,Yes,NaN,NaN,AS,AS,NaN,NaN,<NA>


In [226]:
df_all_quant[df_all_quant['country'] == 'Nigeria']['facility_name'].value_counts(dropna=False)

facility_name
NaN                                                      155
Federal Medical Centre, Birnin Kebbi                     100
University of Maiduguri Teaching Hospital (UMTH)         100
NGH (confirm full name)                                  100
WGH (confirm full name)                                  100
Nnamdi Azikiwe University Teaching Hospital (NAUTH)      100
Alimosho (PHC/Clinic)                                    100
Zankli Medical Services                                   99
UUTH (confirm full name)                                  99
Federal Medical Centre, Keffi                             99
Rivers State University Teaching Hospital (RSUTH)         99
Aminu Kano Teaching Hospital (AKTH)                       96
Ahmadu Bello University Teaching Hospital (ABUTH)         96
UNTH / Uwani Health Centre (Enugu)                        95
Olabisi Onabanjo University Teaching Hospital (OOUTH)     94
Parklane Hospital, Enugu                                  92
GME (confi

In [227]:
df_all_quant.head(2)

,record_id,facility_id,participant_id,country,facility_name,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,"Manhyia District Hospital (MDH), Ghana",At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
1,2,GH001,GH2,Ghana,"Manhyia District Hospital (MDH), Ghana",At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3.0,2024-02-19,Female,Strongly Agree,Disagree,Agree,Strongly Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,AC,AA,AC,NaN,NaN,1


In [228]:
def make_facility_table(
    df,
    country_col="country",
    facility_col="facility_name",
    pct="overall", 
):
    country_order = ["Ghana", "Mali", "Nigeria", "Tanzania", "Uganda", "Zambia", "Zimbabwe"]

    N_total = len(df)
    rows = []

    for c in country_order:
        sub = df[df[country_col].eq(c)]
        if sub.empty:
            continue

        n_country = len(sub)

        if pct == "within_country":
            rows.append({
                "Site facility": c,
                "Frequency, n(%)": f"{n_country} (100.0%)"
            })
        else:
            rows.append({
                "Site facility": c,
                "Frequency, n(%)": f"{n_country} ({100*n_country/N_total:.1f}%)"
            })

        sub_clean = sub[sub[facility_col].notna()]

        if c == "Nigeria":
            fac_counts = (
                sub_clean[facility_col]
                .value_counts()
                .sort_values(ascending=False)
            )
        else:
            fac_counts = (
                sub_clean[facility_col]
                .value_counts()
                .sort_index()
            )

        for fac, n in fac_counts.items():
            denom = n_country if pct == "within_country" else N_total
            rows.append({
                "Site facility": f"  {fac}",
                "Frequency, n(%)": f"{n} ({100*n/denom:.1f}%)"
            })

    return pd.DataFrame(rows, columns=["Site facility", "Frequency, n(%)"])

In [229]:
table_fac = make_facility_table(df_all_quant, pct="overall")
table_fac

,Site facility,"Frequency, n(%)"
0,Ghana,999 (13.3%)
1,"Manhyia District Hospital (MDH), Ghana",999 (13.3%)
2,Mali,1011 (13.5%)
3,"CSCOM Yirimadio (Bamako), Mali",1011 (13.5%)
4,Nigeria,2407 (32.1%)
5,"Federal Medical Centre, Birnin Kebbi",100 (1.3%)
6,University of Maiduguri Teaching Hospital (U...,100 (1.3%)
7,NGH (confirm full name),100 (1.3%)
8,WGH (confirm full name),100 (1.3%)
9,Nnamdi Azikiwe University Teaching Hospital ...,100 (1.3%)


In [230]:
table_fac = make_facility_table(df_all_quant, pct="within_country")
table_fac

,Site facility,"Frequency, n(%)"
0,Ghana,999 (100.0%)
1,"Manhyia District Hospital (MDH), Ghana",999 (100.0%)
2,Mali,1011 (100.0%)
3,"CSCOM Yirimadio (Bamako), Mali",1011 (100.0%)
4,Nigeria,2407 (100.0%)
5,"Federal Medical Centre, Birnin Kebbi",100 (4.2%)
6,University of Maiduguri Teaching Hospital (U...,100 (4.2%)
7,NGH (confirm full name),100 (4.2%)
8,WGH (confirm full name),100 (4.2%)
9,Nnamdi Azikiwe University Teaching Hospital ...,100 (4.2%)


In [231]:
df_all_quant[df_all_quant['country'] == 'Zambia']['facility_id'].value_counts(dropna=False)

facility_id
ZM001    255
ZM002    195
nan      100
Name: count, dtype: int64

In [235]:
doc = Document()
doc.add_heading('Table 3', level=1)

# Add table
table = doc.add_table(rows=1, cols=len(table_fac.columns))
table.style = 'Light Shading Accent 1'

hdr_cells = table.rows[0].cells
for i, col_name in enumerate(table_fac.columns):
    hdr_cells[i].text = str(col_name)

for _, row in table_fac.iterrows():
    row_cells = table.add_row().cells
    for i, value in enumerate(row):
        row_cells[i].text = str(value)

doc.save("Prevalence_tables/table_3.docx")

In [236]:
df_all_quant.shape

(7496, 44)

In [237]:
df_all_quant.head(2)

,record_id,facility_id,participant_id,country,facility_name,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,"Manhyia District Hospital (MDH), Ghana",At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
1,2,GH001,GH2,Ghana,"Manhyia District Hospital (MDH), Ghana",At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3.0,2024-02-19,Female,Strongly Agree,Disagree,Agree,Strongly Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,AC,AA,AC,NaN,NaN,1


In [238]:
df_all_quant['dbspoct_test_results'].value_counts(dropna=False)

dbspoct_test_results
AA               5877
AS               1058
NaN               370
AC                107
SS                 59
Indeterminate       9
SC                  9
CC                  7
Name: count, dtype: int64

In [239]:
df_all_quant['std_poct_results'].value_counts(dropna=False)

std_poct_results
NaN              6394
AA                912
AS                114
AC                 37
CC                 18
SS                  9
Indeterminate       8
SC                  4
Name: count, dtype: int64

In [240]:
import pandas as pd

table = (
    df_all_quant
    .assign(std_poct_results=df_all_quant['std_poct_results'].fillna('Missing'))
    .groupby(['country', 'std_poct_results'])
    .size()
    .unstack(fill_value=0)
)

if 'Missing' not in table.columns:
    table['Missing'] = 0

genotype_cols = [c for c in table.columns if c != 'Missing']
table['Recorded'] = table[genotype_cols].sum(axis=1)

ordered_cols = genotype_cols + ['Recorded', 'Missing']
table = table[ordered_cols]

table['Total'] = table['Recorded'] + table['Missing']

table

std_poct_results,AA,AC,AS,CC,Indeterminate,SC,SS,Recorded,Missing,Total
country,,,,,,,,,,
Ghana,75,6,17,0,2,0,1,101,898,999
Mali,433,30,43,17,5,4,3,535,476,1011
Nigeria,0,0,0,0,0,0,0,0,2407,2407
Tanzania,84,1,14,0,0,0,1,100,900,1000
Uganda,85,0,20,1,1,0,4,111,868,979
Zambia,140,0,15,0,0,0,0,155,395,550
Zimbabwe,95,0,5,0,0,0,0,100,450,550


In [241]:
table = (
    df_all_quant
    .assign(std_poct_results=df_all_quant['dbspoct_test_results'].fillna('Missing'))
    .groupby(['country', 'dbspoct_test_results'])
    .size()
    .unstack(fill_value=0)
)

if 'Missing' not in table.columns:
    table['Missing'] = 0

genotype_cols = [c for c in table.columns if c != 'Missing']
table['Recorded'] = table[genotype_cols].sum(axis=1)

ordered_cols = genotype_cols + ['Recorded', 'Missing']
table = table[ordered_cols]

table['Total'] = table['Recorded'] + table['Missing']

table

dbspoct_test_results,AA,AC,AS,CC,Indeterminate,SC,SS,Recorded,Missing,Total
country,,,,,,,,,,
Ghana,801,6,81,2,7,3,5,905,0,905
Mali,802,97,99,5,0,6,2,1011,0,1011
Nigeria,1912,4,465,0,0,0,26,2407,0,2407
Tanzania,820,0,163,0,0,0,14,997,0,997
Uganda,745,0,150,0,2,0,9,906,0,906
Zambia,377,0,70,0,0,0,3,450,0,450
Zimbabwe,420,0,30,0,0,0,0,450,0,450


In [242]:
df_all = df_all_quant[["country", "dbspoct_test_results"]]
df_all.head()

,country,dbspoct_test_results
0,Ghana,AA
1,Ghana,AA
2,Ghana,AA
3,Ghana,AS
4,Ghana,AA


In [243]:
df_all.shape

(7496, 2)

In [244]:
def make_poct_prevalence_table(df):
    country_order = ["Ghana", "Mali", "Nigeria", "Tanzania", "Uganda", "Zambia", "Zimbabwe"]

    categories = {
        "SCD": ["SS", "SC"],
        "SCT": ["AS"],
        "Normal": ["AA"],
        "Other Hb types": ["AC", "CC"]
    }

    rows = []

    contingency = {cat: [] for cat in categories.keys()}

    overall_total = len(df)

    for country in country_order:
        sub = df[df["country"] == country]
        N = len(sub)

        if N == 0:
            continue

        out = {"Site": country.capitalize()}
        out["Total, n(%)"] = f"{N} ({100*N/overall_total:.1f}%)"

        for cat_label, cat_vals in categories.items():
            n_cat = sub["dbspoct_test_results"].isin(cat_vals).sum()
            pct = 100 * n_cat / N if N > 0 else 0
            out[f"{cat_label} n(%)"] = f"{n_cat} ({pct:.1f}%)"
            contingency[cat_label].append(n_cat)

        rows.append(out)

    obs = []
    for cat in categories.keys():
        obs.append(contingency[cat])
    obs = np.array(obs).T 

    cont = pd.crosstab(df_all['country'], df_all['dbspoct_test_results']).loc[lambda x: x.sum(1).gt(0), lambda x: x.sum(0).gt(0)]
    chi2, p_value, dof, expected = chi2_contingency(cont.values)

    final_rows = []
    for r in rows:
        r["p-value"] = f"{p_value:.4f}"
        final_rows.append(r)

    total_row = {"Site": "Total", "Total, n(%)": f"{overall_total} (100%)"}
    for cat_label, cat_vals in categories.items():
        n_total = df["dbspoct_test_results"].isin(cat_vals).sum()
        pct_total = 100 * n_total / overall_total
        total_row[f"{cat_label} n(%)"] = f"{n_total} ({pct_total:.1f}%)"
    total_row["p-value"] = ""
    final_rows.append(total_row)

    col_order = [
        "Site",
        "Total, n(%)",
        "SCD n(%)",
        "SCT n(%)",
        "Normal n(%)",
        "Other Hb types n(%)",
        "p-value"
    ]

    return pd.DataFrame(final_rows)[col_order]

In [245]:
table5 = make_poct_prevalence_table(df_all)
table5

,Site,"Total, n(%)",SCD n(%),SCT n(%),Normal n(%),Other Hb types n(%),p-value
0,Ghana,999 (13.3%),8 (0.8%),81 (8.1%),801 (80.2%),8 (0.8%),0.0000
1,Mali,1011 (13.5%),8 (0.8%),99 (9.8%),802 (79.3%),102 (10.1%),0.0000
2,Nigeria,2407 (32.1%),26 (1.1%),465 (19.3%),1912 (79.4%),4 (0.2%),0.0000
3,Tanzania,1000 (13.3%),14 (1.4%),163 (16.3%),820 (82.0%),0 (0.0%),0.0000
4,Uganda,979 (13.1%),9 (0.9%),150 (15.3%),745 (76.1%),0 (0.0%),0.0000
5,Zambia,550 (7.3%),3 (0.5%),70 (12.7%),377 (68.5%),0 (0.0%),0.0000
6,Zimbabwe,550 (7.3%),0 (0.0%),30 (5.5%),420 (76.4%),0 (0.0%),0.0000
7,Total,7496 (100%),68 (0.9%),1058 (14.1%),5877 (78.4%),114 (1.5%),


In [246]:
table

dbspoct_test_results,AA,AC,AS,CC,Indeterminate,SC,SS,Recorded,Missing,Total
country,,,,,,,,,,
Ghana,801,6,81,2,7,3,5,905,0,905
Mali,802,97,99,5,0,6,2,1011,0,1011
Nigeria,1912,4,465,0,0,0,26,2407,0,2407
Tanzania,820,0,163,0,0,0,14,997,0,997
Uganda,745,0,150,0,2,0,9,906,0,906
Zambia,377,0,70,0,0,0,3,450,0,450
Zimbabwe,420,0,30,0,0,0,0,450,0,450


In [249]:
doc = Document()
doc.add_heading('Table 5', level=1)

# Add table
table = doc.add_table(rows=1, cols=len(table5.columns))
table.style = 'Light Shading Accent 1'

hdr_cells = table.rows[0].cells
for i, col_name in enumerate(table5.columns):
    hdr_cells[i].text = str(col_name)

for _, row in table5.iterrows():
    row_cells = table.add_row().cells
    for i, value in enumerate(row):
        row_cells[i].text = str(value)

doc.save("Prevalence_tables/table_5_quantitative.docx")

In [250]:
df_all_quant.head(2)

,record_id,facility_id,participant_id,country,facility_name,nbs_screening_locale,date_enrolled,date_of_visit,screening_date,age_guardian,dob_guardian,sex_guardian,tribe_guardian,guardian_relationship,specify_relationship,marital_status_guardian,edu_level_guardian,specify_edu_level_guardian,guardian_occupation,alt_guardian_occupation,religion,mother_children_count,dob_newborn,sex_newborn,detection_at_birth,scd_longevity,early_detection_importance,need_immediate_result,enough_poct_info,poct_result_speed,screening_recommendation,poct_recommendation,dbs_for_validation,proper_treatment,scd_in_the_family,registry_willingness,no_followup_reasons,nofollowup_reason_specified,std_poct_results,dbspoct_test_results,ief_test_results,hplc_results,add_molecula_test_results,age_newborn
0,1,GH001,GH1,Ghana,"Manhyia District Hospital (MDH), Ghana",At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,27,1996-10-12,Female,Akan,Mother,NaN,Cohabiting,Secondary,NaN,Others,Decorator,Christianity,1.0,2024-02-19,Male,Neutral/ I don't know,Disagree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,No,Yes,Yes,NaN,AA,AA,AA,NaN,NaN,1
1,2,GH001,GH2,Ghana,"Manhyia District Hospital (MDH), Ghana",At Birthing Facility,2024-02-20,2024-02-20,20 02 2024,<NA>,NaT,Female,Saliga,Mother,NaN,Married,Secondary,NaN,Others,Seamstress,Islam,3.0,2024-02-19,Female,Strongly Agree,Disagree,Agree,Strongly Agree,Strongly Agree,Neutral/ I don't know,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,I don't know,Yes,Yes,NaN,AC,AA,AC,NaN,NaN,1


In [251]:
df_all_quant[df_all_quant['country'] == 'Zimbabwe']['dbspoct_test_results'].value_counts(dropna=False)

dbspoct_test_results
AA     420
NaN    100
AS      30
Name: count, dtype: int64

In [252]:
df_all_quant[df_all_quant['country'] == 'Zimbabwe']['std_poct_results'].value_counts(dropna=False)

std_poct_results
NaN    450
AA      95
AS       5
Name: count, dtype: int64